# 02 — Recalibrate Simple Signal Grid  
## Locked 2621 bucketed Core / Secondary baseline

This notebook is the cleaned baseline for the SPX VRP simple signal grid.

### Current locked candidate

`locked_2621_win_band_25bps_conditional`

### What this notebook does

1. Loads the production feature panel.
2. Loads the completed naked ATM put outcome panel.
3. Builds the complete-date matrix representation.
4. Evaluates the locked bucketed **Core / Secondary 2621** candidate.
5. Uses the final blended tenor priority:
   - Core first.
   - Secondary only if no Core tenor qualifies.
   - Within the active layer, rank by layer-specific 2621-conditional win probability.
   - Treat tenors within 25 bps of the best conditional win probability as near-ties.
   - Use conditional average P&L/day as the tie-break inside that near-tie set.
6. Saves selected trades, summary tables, thresholds, tenor-priority metrics, and worst-trade audits.

### What was intentionally removed

The executable path no longer includes:

- broad grid sweeps
- pair 144 intermediate searches
- old P&L/day-only tenor priority logic
- conditional priority bakeoff code
- 2026 stress-diagnostic exploratory code
- line-fitting / smoothing experiments

Those steps were useful research, but this file is now meant to be a stable rerunnable baseline.


In [ ]:
# Cell 1 — Setup, imports, paths, and project metadata

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
# This notebook is intended to run from either the project root or a
# notebooks/ subfolder. If neither location has a data/ directory, it falls
# back to the project path used during development.
# ---------------------------------------------------------------------

cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PANEL_PATH = PROCESSED_DIR / "production_feature_panel_v0_1.parquet"
OUTCOME_PATH = PROCESSED_DIR / "naked_atm_put_eod_panel_v0_1.parquet"

# ---------------------------------------------------------------------
# Tenor universe and grouping
# ---------------------------------------------------------------------

PRODUCTION_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

GROUP_TENORS = {
    "front": [9, 12, 15],
    "middle": [18, 21, 24],
    "back": [27, 30, 33],
}

TENOR_GROUP_MAP = {
    tenor: group
    for group, tenors in GROUP_TENORS.items()
    for tenor in tenors
}

# ---------------------------------------------------------------------
# Locked model metadata
# ---------------------------------------------------------------------
# Current preferred research candidate:
#   locked_2621_win_band_25bps_conditional
#
# Important conventions:
#   - Thresholds are bucketed front / middle / back, not line-fitted.
#   - Core is checked before Secondary.
#   - Secondary is checked only when no Core tenor qualifies.
#   - Within the active layer, tenor priority is based on layer-specific
#     2621-conditional win probability.
#   - Tenors within 25 bps of the best conditional win probability are treated
#     as near-ties; conditional average P&L/day chooses among that near-tie set.
#   - Raw average P&L is not used for tenor priority.
# ---------------------------------------------------------------------

MODEL_LABEL = "locked_2621_win_band_25bps_conditional"
WIN_BAND = 0.0025

print("Current working directory:", cwd)
print("Project root:", PROJECT_ROOT)
print("Feature panel path:", FEATURE_PANEL_PATH)
print("Outcome path:", OUTCOME_PATH)
print("Feature panel exists:", FEATURE_PANEL_PATH.exists())
print("Outcome file exists:", OUTCOME_PATH.exists())
print("Model label:", MODEL_LABEL)
print("Win band:", WIN_BAND)


In [ ]:
# Cell 2 — Small reusable data-cleaning helpers

def get_col(df: pd.DataFrame, candidates, required=True, label=None):
    """
    Return the first matching column from a candidate list.

    Matching is case-insensitive, but the original DataFrame column name is
    returned. This keeps the notebook robust to small naming changes in source
    files.
    """
    lower_map = {str(c).lower(): c for c in df.columns}

    for c in candidates:
        if c in df.columns:
            return c

        c_lower = str(c).lower()
        if c_lower in lower_map:
            return lower_map[c_lower]

    if required:
        raise KeyError(
            f"Missing column for {label or candidates}. "
            f"Tried: {candidates}. Available columns: {list(df.columns)}"
        )

    return None


def parse_project_date_series(s: pd.Series) -> pd.Series:
    """
    Parse project date columns safely.

    Important:
      Numeric yyyymmdd values such as 20190805 must be parsed as calendar
      dates, not as nanoseconds since 1970.
    """
    s = s.copy()

    if pd.api.types.is_datetime64_any_dtype(s):
        return pd.to_datetime(s)

    s_str = s.astype("string").str.strip()
    s_str = s_str.str.replace(r"\.0$", "", regex=True)

    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    is_yyyymmdd = s_str.str.fullmatch(r"\d{8}", na=False)

    if is_yyyymmdd.any():
        parsed.loc[is_yyyymmdd] = pd.to_datetime(
            s_str.loc[is_yyyymmdd],
            format="%Y%m%d",
            errors="coerce",
        )

    remaining = ~is_yyyymmdd

    if remaining.any():
        parsed.loc[remaining] = pd.to_datetime(
            s_str.loc[remaining],
            errors="coerce",
        )

    return parsed


def clean_binary_series(s: pd.Series) -> pd.Series:
    """
    Convert bool / numeric / string binary values into float 0/1.
    Missing or unresolved values stay NaN.
    """
    if pd.api.types.is_bool_dtype(s):
        return s.astype(float)

    numeric = pd.to_numeric(s, errors="coerce")

    if numeric.notna().mean() > 0.95:
        return numeric.astype(float)

    return (
        s.astype("string")
        .str.lower()
        .str.strip()
        .map({
            "true": 1.0,
            "false": 0.0,
            "yes": 1.0,
            "no": 0.0,
            "1": 1.0,
            "0": 0.0,
            "1.0": 1.0,
            "0.0": 0.0,
        })
    )


def require_columns(df: pd.DataFrame, required_cols, label="DataFrame"):
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"{label} is missing required columns: {missing_cols}")

In [ ]:
# Cell 3 — Load and validate the production feature panel

if not FEATURE_PANEL_PATH.exists():
    raise FileNotFoundError(f"Missing feature panel: {FEATURE_PANEL_PATH}")

features = pd.read_parquet(FEATURE_PANEL_PATH)
features["date"] = pd.to_datetime(features["date"]).dt.normalize()
features = features.sort_values(["date", "tenor"]).reset_index(drop=True)

required_feature_cols = [
    "date",
    "tenor",
    "tenor_group",
    "spx_close",
    "implied_variance",
    "forecast_variance",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
]

require_columns(features, required_feature_cols, label="Feature panel")

features = features[features["tenor"].isin(PRODUCTION_TENORS)].copy()
features["tenor"] = features["tenor"].astype(int)

eligible = features.dropna(
    subset=[
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "implied_variance",
        "forecast_variance",
    ]
).copy()

eligible = eligible.sort_values(["date", "tenor"]).reset_index(drop=True)

dupes = eligible.duplicated(subset=["date", "tenor"]).sum()

print("Loaded feature panel")
print("  Shape:", features.shape)
print("  Date range:", features["date"].min(), "to", features["date"].max())
print("  Tenors:", sorted(features["tenor"].dropna().unique().tolist()))

print("\nEligible feature panel")
print("  Shape:", eligible.shape)
print("  Date range:", eligible["date"].min(), "to", eligible["date"].max())
print("  Tenors:", sorted(eligible["tenor"].dropna().unique().tolist()))
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        eligible[
            eligible.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Duplicate date/tenor rows found in eligible feature panel.")

# rv21d and rsi14 are market-level filters, so they should be identical
# across all tenors on a given date.
market_filter_counts = (
    eligible
    .groupby("date")[["rv21d", "rsi14"]]
    .nunique(dropna=False)
)

bad_market_filter_dates = market_filter_counts[
    (market_filter_counts["rv21d"] > 1) |
    (market_filter_counts["rsi14"] > 1)
]

print("  Dates with inconsistent rv21d/rsi14 across tenors:", len(bad_market_filter_dates))

if len(bad_market_filter_dates) > 0:
    display(bad_market_filter_dates.head(20))
    raise ValueError("rv21d or rsi14 differs across tenors on the same date.")

display(eligible.head(10))

In [ ]:
# Cell 4 — Load naked ATM put outcomes and build the completed research panel

if not OUTCOME_PATH.exists():
    raise FileNotFoundError(f"Missing outcome file: {OUTCOME_PATH}")

outcomes_raw = pd.read_parquet(OUTCOME_PATH)

# ---------------------------------------------------------------------
# Column mapping
# ---------------------------------------------------------------------

date_col = get_col(
    outcomes_raw,
    ["trade_dt", "trade_date", "date", "timestamp", "datetime"],
    label="date",
)

tenor_col = get_col(
    outcomes_raw,
    ["entry_tenor", "tenor", "target_tenor", "actual_dte", "dte"],
    label="tenor",
)

win_col = get_col(
    outcomes_raw,
    ["win_clean", "win", "test_win"],
    label="win",
)

expired_otm_col = get_col(
    outcomes_raw,
    ["expired_otm_clean", "expired_otm"],
    required=False,
    label="expired_otm",
)

pnl_dollars_col = get_col(
    outcomes_raw,
    ["normalized_pnl_dollars", "test_pnl", "sized_pnl"],
    required=False,
    label="normalized_pnl_dollars",
)

pnl_pct_col = get_col(
    outcomes_raw,
    ["normalized_pnl_pct"],
    required=False,
    label="normalized_pnl_pct",
)

actual_dte_col = get_col(
    outcomes_raw,
    ["actual_dte"],
    required=False,
    label="actual_dte",
)

expiry_spx_col = get_col(
    outcomes_raw,
    ["expiry_spx_close"],
    required=False,
    label="expiry_spx_close",
)

entry_credit_col = get_col(
    outcomes_raw,
    ["entry_credit_points", "atm_put_mid"],
    required=False,
    label="entry_credit_points",
)

short_pnl_points_col = get_col(
    outcomes_raw,
    ["short_option_pnl_points"],
    required=False,
    label="short_option_pnl_points",
)

print("Outcome column mapping:")
print({
    "date_col": date_col,
    "tenor_col": tenor_col,
    "win_col": win_col,
    "expired_otm_col": expired_otm_col,
    "pnl_dollars_col": pnl_dollars_col,
    "pnl_pct_col": pnl_pct_col,
    "actual_dte_col": actual_dte_col,
    "expiry_spx_col": expiry_spx_col,
    "entry_credit_col": entry_credit_col,
    "short_pnl_points_col": short_pnl_points_col,
})

# ---------------------------------------------------------------------
# Standardize outcomes
# ---------------------------------------------------------------------

outcomes = pd.DataFrame({
    "date": parse_project_date_series(outcomes_raw[date_col]).dt.normalize(),
    "tenor": pd.to_numeric(outcomes_raw[tenor_col], errors="coerce").astype("Int64"),
    "win_clean": clean_binary_series(outcomes_raw[win_col]),
})

if expired_otm_col is not None:
    outcomes["expired_otm_clean"] = clean_binary_series(outcomes_raw[expired_otm_col])
else:
    outcomes["expired_otm_clean"] = np.nan

if pnl_dollars_col is not None:
    outcomes["normalized_pnl_dollars"] = pd.to_numeric(outcomes_raw[pnl_dollars_col], errors="coerce")
else:
    outcomes["normalized_pnl_dollars"] = np.nan

if pnl_pct_col is not None:
    outcomes["normalized_pnl_pct"] = pd.to_numeric(outcomes_raw[pnl_pct_col], errors="coerce")
else:
    outcomes["normalized_pnl_pct"] = np.nan

if actual_dte_col is not None:
    outcomes["actual_dte"] = pd.to_numeric(outcomes_raw[actual_dte_col], errors="coerce")
else:
    outcomes["actual_dte"] = np.nan

if expiry_spx_col is not None:
    outcomes["expiry_spx_close"] = pd.to_numeric(outcomes_raw[expiry_spx_col], errors="coerce")
else:
    outcomes["expiry_spx_close"] = np.nan

if entry_credit_col is not None:
    outcomes["entry_credit_points"] = pd.to_numeric(outcomes_raw[entry_credit_col], errors="coerce")
else:
    outcomes["entry_credit_points"] = np.nan

if short_pnl_points_col is not None:
    outcomes["short_option_pnl_points"] = pd.to_numeric(outcomes_raw[short_pnl_points_col], errors="coerce")
else:
    outcomes["short_option_pnl_points"] = np.nan

outcomes = outcomes[outcomes["tenor"].isin(PRODUCTION_TENORS)].copy()
outcomes = outcomes.dropna(subset=["date", "tenor"]).copy()
outcomes["tenor"] = outcomes["tenor"].astype(int)

dupes = outcomes.duplicated(subset=["date", "tenor"]).sum()

print("\nStandardized outcome panel")
print("  Shape:", outcomes.shape)
print("  Date range:", outcomes["date"].min(), "to", outcomes["date"].max())
print("  Tenors:", sorted(outcomes["tenor"].dropna().unique().tolist()))
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        outcomes[
            outcomes.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Outcome panel has duplicate date/tenor rows.")

# ---------------------------------------------------------------------
# Join outcomes to eligible features.
# ---------------------------------------------------------------------
#
# Missing outcomes near the right edge are expected because later-dated
# longer-tenor trades may not have expired yet. Those rows are dropped.
#
# Missing outcomes before the last completed outcome date for that tenor are
# treated as a data hole and raise an error.
# ---------------------------------------------------------------------

research_panel_raw = eligible.merge(
    outcomes,
    on=["date", "tenor"],
    how="left",
)

completed_outcomes = outcomes.dropna(subset=["win_clean"]).copy()

last_completed_date_by_tenor = (
    completed_outcomes
    .groupby("tenor")["date"]
    .max()
    .rename("last_completed_outcome_date")
    .reset_index()
)

missing_outcomes = (
    research_panel_raw[research_panel_raw["win_clean"].isna()]
    [["date", "tenor", "tenor_group", "spx_close", "vrp_log", "vrp_z_3m", "vrp_z_1y", "rv21d", "rsi14"]]
    .copy()
)

missing_outcomes = missing_outcomes.merge(
    last_completed_date_by_tenor,
    on="tenor",
    how="left",
)

unexpected_missing = missing_outcomes[
    missing_outcomes["date"] <= missing_outcomes["last_completed_outcome_date"]
].copy()

print("\nResearch panel after outcome join")
print("  Shape:", research_panel_raw.shape)
print("  Rows with missing win_clean:", int(research_panel_raw["win_clean"].isna().sum()))

print("\nLast completed outcome date by tenor:")
display(last_completed_date_by_tenor)

if len(missing_outcomes) > 0:
    missing_summary = (
        missing_outcomes
        .groupby("tenor")
        .agg(
            missing_rows=("date", "size"),
            first_missing_date=("date", "min"),
            last_missing_date=("date", "max"),
            last_completed_outcome_date=("last_completed_outcome_date", "max"),
        )
        .reset_index()
    )

    print("\nMissing outcome summary by tenor:")
    display(missing_summary)

if len(unexpected_missing) > 0:
    print("\nUnexpected missing outcomes found before/equal to last completed date for that tenor:")
    display(unexpected_missing.head(30))
    raise ValueError("Unexpected non-right-edge missing outcomes found.")

print("\nRight-edge censored rows to drop:", len(missing_outcomes))

research_panel = research_panel_raw.dropna(subset=["win_clean"]).copy()
research_panel = research_panel.sort_values(["date", "tenor"]).reset_index(drop=True)

required_outcome_cols = ["win_clean", "normalized_pnl_dollars", "actual_dte"]
require_columns(research_panel, required_outcome_cols, label="Completed research panel")

if research_panel[required_outcome_cols].isna().any().any():
    display(research_panel[research_panel[required_outcome_cols].isna().any(axis=1)].head(20))
    raise ValueError("Completed research panel has missing win/P&L/actual_dte values.")

print("\nFinal completed research panel")
print("  Shape:", research_panel.shape)
print("  Date range:", research_panel["date"].min(), "to", research_panel["date"].max())
print("  Tenors:", sorted(research_panel["tenor"].dropna().unique().tolist()))
print("  Overall win rate:", research_panel["win_clean"].mean())

outcome_by_tenor = (
    research_panel
    .groupby("tenor")
    .agg(
        rows=("win_clean", "size"),
        win_rate=("win_clean", "mean"),
        avg_pnl_dollars=("normalized_pnl_dollars", "mean"),
        avg_pnl_pct=("normalized_pnl_pct", "mean"),
        avg_actual_dte=("actual_dte", "mean"),
    )
    .reset_index()
)

print("\nOutcome summary by tenor:")
display(outcome_by_tenor)

In [ ]:
# Cell 5 — Build complete-date sweep panel and matrix representation

# ---------------------------------------------------------------------
# Outcome normalization
# ---------------------------------------------------------------------
# The selected-tenor rule uses conditional P&L/day as a tie-breaker. This is
# not the same as using raw average P&L, which mechanically favors longer DTE.
# ---------------------------------------------------------------------

research_panel = research_panel.copy()

with np.errstate(divide="ignore", invalid="ignore"):
    research_panel["outcome_pnl_per_day"] = (
        research_panel["normalized_pnl_dollars"] / research_panel["actual_dte"]
    )

research_panel.loc[
    ~np.isfinite(research_panel["outcome_pnl_per_day"]),
    "outcome_pnl_per_day",
] = np.nan

# ---------------------------------------------------------------------
# Sweep panel cleaning
# ---------------------------------------------------------------------
# Keep only the columns needed for the locked signal evaluation and future
# overlay/sizing work.
# ---------------------------------------------------------------------

sweep_panel = research_panel[
    [
        "date",
        "tenor",
        "tenor_group",
        "spx_close",
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "win_clean",
        "normalized_pnl_dollars",
        "actual_dte",
        "outcome_pnl_per_day",
    ]
].copy()

numeric_cols = [
    "tenor",
    "spx_close",
    "vrp_log",
    "vrp_z_3m",
    "vrp_z_1y",
    "rv21d",
    "rsi14",
    "win_clean",
    "normalized_pnl_dollars",
    "actual_dte",
    "outcome_pnl_per_day",
]

for col in numeric_cols:
    sweep_panel[col] = pd.to_numeric(sweep_panel[col], errors="coerce")

before_drop = len(sweep_panel)

sweep_panel = sweep_panel.dropna(
    subset=[
        "date",
        "tenor",
        "tenor_group",
        "spx_close",
        "vrp_log",
        "vrp_z_3m",
        "vrp_z_1y",
        "rv21d",
        "rsi14",
        "win_clean",
        "normalized_pnl_dollars",
        "actual_dte",
        "outcome_pnl_per_day",
    ]
).copy()

after_drop = len(sweep_panel)

print("\nSweep panel row cleaning")
print("  Rows before drop:", before_drop)
print("  Rows after drop: ", after_drop)
print("  Rows dropped:     ", before_drop - after_drop)

dupes = sweep_panel.duplicated(subset=["date", "tenor"]).sum()
print("  Duplicate date/tenor rows:", dupes)

if dupes > 0:
    display(
        sweep_panel[
            sweep_panel.duplicated(subset=["date", "tenor"], keep=False)
        ].sort_values(["date", "tenor"]).head(30)
    )
    raise ValueError("Duplicate date/tenor rows in sweep_panel.")

# ---------------------------------------------------------------------
# Complete-date filter
# ---------------------------------------------------------------------
# Keep only dates where all 9 tenors have completed outcomes. This keeps trade
# frequency comparisons clean and prevents right-edge censoring from biasing
# tenor selection.
# ---------------------------------------------------------------------

REQUIRED_TENOR_COUNT = len(PRODUCTION_TENORS)

tenor_count_by_date = (
    sweep_panel
    .groupby("date")["tenor"]
    .nunique()
    .rename("tenor_count")
)

complete_dates = tenor_count_by_date[tenor_count_by_date == REQUIRED_TENOR_COUNT].index
incomplete_dates = tenor_count_by_date[tenor_count_by_date != REQUIRED_TENOR_COUNT]

print("\nComplete-date filter")
print("  Original sweep rows:", len(sweep_panel))
print("  Original sweep dates:", sweep_panel["date"].nunique())
print("  Complete dates:", len(complete_dates))
print("  Incomplete dates:", len(incomplete_dates))

if len(incomplete_dates) > 0:
    print("\nIncomplete date summary:")
    display(
        incomplete_dates
        .reset_index()
        .sort_values("date")
    )

sweep_panel = sweep_panel[sweep_panel["date"].isin(complete_dates)].copy()
sweep_panel = sweep_panel.sort_values(["date", "tenor"]).reset_index(drop=True)

sweep_dates = pd.Index(sorted(sweep_panel["date"].unique()))
num_sweep_dates = len(sweep_dates)
TRADE_FREQUENCY_DENOMINATOR = num_sweep_dates

print("\nClean sweep panel")
print("  Rows:", len(sweep_panel))
print("  Dates:", num_sweep_dates)
print("  Date range:", sweep_dates.min(), "to", sweep_dates.max())
print("  Trade frequency denominator:", TRADE_FREQUENCY_DENOMINATOR)

clean_tenor_count_by_date = sweep_panel.groupby("date")["tenor"].nunique()

if not (clean_tenor_count_by_date == REQUIRED_TENOR_COUNT).all():
    raise ValueError("Some remaining dates do not have all 9 tenors.")

print("\nRows by tenor group after complete-date filter:")
display(
    sweep_panel
    .groupby("tenor_group")
    .agg(
        rows=("win_clean", "size"),
        win_rate=("win_clean", "mean"),
        avg_vrp=("vrp_log", "mean"),
        avg_z3m=("vrp_z_3m", "mean"),
        avg_z1y=("vrp_z_1y", "mean"),
        avg_rv21d=("rv21d", "mean"),
        avg_rsi14=("rsi14", "mean"),
        avg_pnl_per_day=("outcome_pnl_per_day", "mean"),
        total_pnl_dollars=("normalized_pnl_dollars", "sum"),
        total_actual_dte=("actual_dte", "sum"),
    )
    .assign(
        aggregate_pnl_per_day=lambda x: x["total_pnl_dollars"] / x["total_actual_dte"]
    )
    .reset_index()
)

# ---------------------------------------------------------------------
# Matrix representation
# ---------------------------------------------------------------------
# All strategy logic below uses matrix arrays:
#   rows = dates
#   columns = tenors [9, 12, ..., 33]
# ---------------------------------------------------------------------

TENOR_ARRAY = np.array(PRODUCTION_TENORS, dtype=int)
TENOR_ORDER = TENOR_ARRAY.tolist()
TENOR_TO_COL = {int(t): i for i, t in enumerate(TENOR_ARRAY)}

GROUP_COLS = {
    group: [TENOR_TO_COL[t] for t in tenors]
    for group, tenors in GROUP_TENORS.items()
}


def pivot_to_matrix(value_col):
    mat_df = (
        sweep_panel
        .pivot(index="date", columns="tenor", values=value_col)
        .reindex(index=sweep_dates, columns=PRODUCTION_TENORS)
    )

    if mat_df.isna().any().any():
        bad = mat_df[mat_df.isna().any(axis=1)].head(10)
        display(bad)
        raise ValueError(f"Matrix for {value_col} contains missing values.")

    return mat_df.to_numpy()


spx_mat = pivot_to_matrix("spx_close")
spx_close_mat = spx_mat
vrp_mat = pivot_to_matrix("vrp_log")
z3m_mat = pivot_to_matrix("vrp_z_3m")
z1y_mat = pivot_to_matrix("vrp_z_1y")
win_mat = pivot_to_matrix("win_clean")
pnl_mat = pivot_to_matrix("normalized_pnl_dollars")
actual_dte_mat = pivot_to_matrix("actual_dte")
pnl_per_day_mat = pivot_to_matrix("outcome_pnl_per_day")

rv21d_by_date = (
    sweep_panel.groupby("date")["rv21d"].first().reindex(sweep_dates).to_numpy()
)

rsi_by_date = (
    sweep_panel.groupby("date")["rsi14"].first().reindex(sweep_dates).to_numpy()
)

print("\nMatrix build complete")
print("  Matrix shape:", vrp_mat.shape)
print("  Tenor order:", TENOR_ORDER)


## Locked model specification

### Threshold surface

The production research candidate uses **bucketed** front / middle / back thresholds. Do **not** line-fit or smooth these thresholds in this baseline notebook.

| Layer | Parameter | Front | Middle | Back |
|---|---|---:|---:|---:|
| Core | VRP | 0.60 | 0.65 | 0.70 |
| Core | z3m | 0.55 | 0.75 | 0.75 |
| Core | z1y | 0.65 | 0.65 | 0.75 |
| Core | RSI cap | 70 | 68 | 66 |
| Core | RV21D floor | 8.5 | 8.5 | 8.5 |
| Secondary | VRP | 0.60 | 0.60 | 0.70 |
| Secondary | z3m | 0.50 | 0.50 | 0.50 |
| Secondary | z1y | 0.40 | 0.50 | 0.50 |
| Secondary | RSI cap | 74 | 70 | 68 |
| Secondary | RV21D floor | 6.5 | 6.5 | 6.5 |

### Selection convention

1. Evaluate all Core tenors.
2. If any Core tenor qualifies, choose one Core tenor and ignore Secondary.
3. If no Core tenor qualifies, evaluate all Secondary tenors and choose one Secondary tenor if any qualify.
4. Maximum one selected trade per date.
5. Tenor priority is conditional on the locked 2621 qualifying universe, not unconditional tenor statistics.


In [ ]:
# Cell — Final locked 2621 blended tenor-priority model
#
# Final model convention:
#   1. Thresholds are locked to 2621.
#   2. Core is checked first.
#   3. Secondary is checked only if no Core tenor qualifies.
#   4. Within the active layer:
#        a. Rank tenors by layer-specific 2621-conditional win probability.
#        b. Keep tenors within 25 bps of the best qualifying tenor's win probability.
#        c. Select the tenor with highest layer-specific 2621-conditional avg P&L/day.
#        d. Tie-break by aggregate P&L/day, then conditional win probability, then longer tenor.
#
# Why:
#   - Avoids raw P&L duration bias.
#   - Avoids unconditional tenor statistics.
#   - Avoids pure P&L/day selection taking over the model.
#   - Keeps win probability as the primary criterion.
#
# Expected tenor behavior from bakeoff:
#   - Same front/middle/back mix as win_only.
#   - Main change: inside the back bucket, 27D can beat 33D when their conditional
#     win probabilities are effectively tied and 27D has better P&L/day.

import numpy as np
import pandas as pd
import json
from pathlib import Path

# ---------------------------------------------------------------------
# Robust setup
# ---------------------------------------------------------------------

if "TENOR_ORDER" in globals():
    TENOR_ARRAY = np.array(TENOR_ORDER, dtype=int)

elif "sweep_panel" in globals() and "tenor" in sweep_panel.columns:
    TENOR_ARRAY = np.array(sorted(sweep_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

elif "research_panel" in globals() and "tenor" in research_panel.columns:
    TENOR_ARRAY = np.array(sorted(research_panel["tenor"].dropna().unique()), dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

else:
    TENOR_ARRAY = np.array([9, 12, 15, 18, 21, 24, 27, 30, 33], dtype=int)
    TENOR_ORDER = TENOR_ARRAY.tolist()

TENOR_TO_COL = {int(t): i for i, t in enumerate(TENOR_ARRAY)}

required_objects = [
    "num_sweep_dates",
    "sweep_dates",
    "vrp_mat",
    "z3m_mat",
    "z1y_mat",
    "win_mat",
    "pnl_mat",
    "actual_dte_mat",
    "rv21d_by_date",
    "rsi_by_date",
    "TRADE_FREQUENCY_DENOMINATOR",
]

missing_objects = [name for name in required_objects if name not in globals()]

if missing_objects:
    raise NameError(
        "Missing required objects. Run the data-loading / matrix-construction cells first. "
        f"Missing: {missing_objects}"
    )

if "pnl_per_day_mat" not in globals():
    with np.errstate(divide="ignore", invalid="ignore"):
        pnl_per_day_mat = pnl_mat / actual_dte_mat
    pnl_per_day_mat[~np.isfinite(pnl_per_day_mat)] = np.nan

if "AUDIT_DIR" not in globals():
    AUDIT_DIR = Path("data/audit")
else:
    AUDIT_DIR = Path(AUDIT_DIR)

if "PROCESSED_DIR" not in globals():
    PROCESSED_DIR = Path("data/processed")
else:
    PROCESSED_DIR = Path(PROCESSED_DIR)

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_LABEL = "locked_2621_win_band_25bps_conditional"
WIN_BAND = 0.0025

print("Final locked 2621 blended-priority model")
print("Model label:", MODEL_LABEL)
print("Tenors:", TENOR_ARRAY.tolist())
print("Win band:", WIN_BAND)


# ---------------------------------------------------------------------
# Locked 2621 thresholds
# ---------------------------------------------------------------------

LOCKED_2621_CORE = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.55,
        "z1y": 0.65,
        "rsi_cap": 70,
        "rv21d_floor": 8.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.65,
        "z3m": 0.75,
        "z1y": 0.65,
        "rsi_cap": 68,
        "rv21d_floor": 8.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.75,
        "z1y": 0.75,
        "rsi_cap": 66,
        "rv21d_floor": 8.5,
    },
}

LOCKED_2621_SECONDARY = {
    "front": {
        "tenors": [9, 12, 15],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.40,
        "rsi_cap": 74,
        "rv21d_floor": 6.5,
    },
    "middle": {
        "tenors": [18, 21, 24],
        "vrp": 0.60,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 70,
        "rv21d_floor": 6.5,
    },
    "back": {
        "tenors": [27, 30, 33],
        "vrp": 0.70,
        "z3m": 0.50,
        "z1y": 0.50,
        "rsi_cap": 68,
        "rv21d_floor": 6.5,
    },
}


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def tenor_group_for_tenor(tenor):
    tenor = int(tenor)

    if tenor in [9, 12, 15]:
        return "front"
    elif tenor in [18, 21, 24]:
        return "middle"
    elif tenor in [27, 30, 33]:
        return "back"
    else:
        raise ValueError(f"Unexpected tenor: {tenor}")


def build_threshold_table(threshold_spec, layer):
    rows = []

    for group, params in threshold_spec.items():
        for tenor in params["tenors"]:
            rows.append({
                "layer": layer,
                "tenor_group": group,
                "selected_tenor": int(tenor),
                "vrp_threshold": float(params["vrp"]),
                "z3m_threshold": float(params["z3m"]),
                "z1y_threshold": float(params["z1y"]),
                "rsi_cap": float(params["rsi_cap"]),
                "rv21d_floor": float(params["rv21d_floor"]),
            })

    return pd.DataFrame(rows)


def build_qualifies_matrix(threshold_spec):
    """
    Builds date x tenor boolean qualification matrix for locked 2621 thresholds.
    """
    qualifies = np.zeros((num_sweep_dates, len(TENOR_ARRAY)), dtype=bool)

    for col, tenor in enumerate(TENOR_ARRAY):
        group = tenor_group_for_tenor(tenor)
        params = threshold_spec[group]

        qualifies[:, col] = (
            (vrp_mat[:, col] >= float(params["vrp"])) &
            (z3m_mat[:, col] >= float(params["z3m"])) &
            (z1y_mat[:, col] >= float(params["z1y"])) &
            (rsi_by_date <= float(params["rsi_cap"])) &
            (rv21d_by_date >= float(params["rv21d_floor"]))
        )

    return qualifies


def build_layer_conditional_tenor_metrics(layer_name, qualifies_matrix, date_mask):
    """
    Builds tenor ranking metrics conditional on the relevant 2621 candidate universe.

    Core:
      date_mask = all dates.

    Secondary:
      date_mask = dates where no Core tenor qualifies.
    """
    rows = []

    for col, tenor in enumerate(TENOR_ARRAY):
        candidate_mask = (
            date_mask &
            qualifies_matrix[:, col] &
            np.isfinite(win_mat[:, col]) &
            np.isfinite(pnl_mat[:, col]) &
            np.isfinite(actual_dte_mat[:, col]) &
            np.isfinite(pnl_per_day_mat[:, col])
        )

        candidate_count = int(candidate_mask.sum())

        if candidate_count == 0:
            rows.append({
                "layer": layer_name,
                "selected_col": int(col),
                "selected_tenor": int(tenor),
                "tenor_group": tenor_group_for_tenor(tenor),
                "conditional_win_probability": np.nan,
                "conditional_avg_pnl_per_day": np.nan,
                "conditional_median_pnl_per_day": np.nan,
                "conditional_aggregate_pnl_per_day": np.nan,
                "conditional_avg_raw_pnl": np.nan,
                "conditional_avg_actual_dte": np.nan,
                "conditional_worst_trade_pnl": np.nan,
                "conditional_worst_pnl_per_day": np.nan,
                "candidate_count": 0,
            })
            continue

        rows.append({
            "layer": layer_name,
            "selected_col": int(col),
            "selected_tenor": int(tenor),
            "tenor_group": tenor_group_for_tenor(tenor),

            "conditional_win_probability": float(np.nanmean(win_mat[candidate_mask, col])),
            "conditional_avg_pnl_per_day": float(np.nanmean(pnl_per_day_mat[candidate_mask, col])),
            "conditional_median_pnl_per_day": float(np.nanmedian(pnl_per_day_mat[candidate_mask, col])),
            "conditional_aggregate_pnl_per_day": float(
                np.nansum(pnl_mat[candidate_mask, col]) /
                np.nansum(actual_dte_mat[candidate_mask, col])
            ),
            "conditional_avg_raw_pnl": float(np.nanmean(pnl_mat[candidate_mask, col])),
            "conditional_avg_actual_dte": float(np.nanmean(actual_dte_mat[candidate_mask, col])),
            "conditional_worst_trade_pnl": float(np.nanmin(pnl_mat[candidate_mask, col])),
            "conditional_worst_pnl_per_day": float(np.nanmin(pnl_per_day_mat[candidate_mask, col])),
            "candidate_count": candidate_count,
        })

    return pd.DataFrame(rows)


def blended_priority_order(layer_name, qualifying_cols, win_band=WIN_BAND):
    """
    Final blended priority rule.

    Step 1:
      Find best conditional win probability among qualifying tenors.

    Step 2:
      Keep tenors within win_band of that best conditional win probability.

    Step 3:
      Select highest conditional avg P&L/day.
      Tie-break by aggregate P&L/day, then conditional win probability, then longer tenor.
    """
    if len(qualifying_cols) == 0:
        return pd.DataFrame()

    df = tenor_priority_metrics[
        (tenor_priority_metrics["layer"].eq(layer_name)) &
        (tenor_priority_metrics["selected_col"].isin(qualifying_cols)) &
        (tenor_priority_metrics["candidate_count"] > 0)
    ].copy()

    if df.empty:
        return df

    best_win = df["conditional_win_probability"].max()

    df["best_conditional_win_probability"] = best_win
    df["win_probability_gap_to_best"] = best_win - df["conditional_win_probability"]
    df["inside_win_band"] = df["win_probability_gap_to_best"] <= win_band

    df = df[df["inside_win_band"]].copy()

    df = df.sort_values(
        [
            "conditional_avg_pnl_per_day",
            "conditional_aggregate_pnl_per_day",
            "conditional_win_probability",
            "selected_tenor",
        ],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)

    df["priority_rank_within_signal"] = np.arange(1, len(df) + 1)

    return df


def select_final_model_path():
    """
    Core-first / Secondary-second final path selection.
    """
    selected_rows = []

    core_selected = np.full(num_sweep_dates, -1, dtype=np.int8)
    secondary_selected = np.full(num_sweep_dates, -1, dtype=np.int8)
    combined_selected = np.full(num_sweep_dates, -1, dtype=np.int8)

    for row in range(num_sweep_dates):
        core_qualifying_cols = np.where(core_qualifies_matrix[row, :])[0].astype(int).tolist()

        if core_qualifying_cols:
            ordered = blended_priority_order(
                layer_name="Core",
                qualifying_cols=core_qualifying_cols,
                win_band=WIN_BAND,
            )

            if not ordered.empty:
                selected_col = int(ordered.iloc[0]["selected_col"])

                core_selected[row] = selected_col
                combined_selected[row] = selected_col

                selected_rows.append({
                    "row_idx": int(row),
                    "date": sweep_dates[row],
                    "layer": "Core",
                    "selected_col": selected_col,
                    "selected_tenor": int(TENOR_ARRAY[selected_col]),
                    "tenor_group": tenor_group_for_tenor(TENOR_ARRAY[selected_col]),
                    "num_qualifying_tenors_in_layer": int(len(core_qualifying_cols)),
                    "qualifying_tenors_in_layer": ",".join(
                        [str(int(TENOR_ARRAY[c])) for c in core_qualifying_cols]
                    ),
                    "selection_rule": "win_band_25bps_then_pnl_day",
                    "win_band": WIN_BAND,
                })

            continue

        secondary_qualifying_cols = np.where(secondary_qualifies_matrix[row, :])[0].astype(int).tolist()

        if secondary_qualifying_cols:
            ordered = blended_priority_order(
                layer_name="Secondary",
                qualifying_cols=secondary_qualifying_cols,
                win_band=WIN_BAND,
            )

            if not ordered.empty:
                selected_col = int(ordered.iloc[0]["selected_col"])

                secondary_selected[row] = selected_col
                combined_selected[row] = selected_col

                selected_rows.append({
                    "row_idx": int(row),
                    "date": sweep_dates[row],
                    "layer": "Secondary",
                    "selected_col": selected_col,
                    "selected_tenor": int(TENOR_ARRAY[selected_col]),
                    "tenor_group": tenor_group_for_tenor(TENOR_ARRAY[selected_col]),
                    "num_qualifying_tenors_in_layer": int(len(secondary_qualifying_cols)),
                    "qualifying_tenors_in_layer": ",".join(
                        [str(int(TENOR_ARRAY[c])) for c in secondary_qualifying_cols]
                    ),
                    "selection_rule": "win_band_25bps_then_pnl_day",
                    "win_band": WIN_BAND,
                })

    selected_path = pd.DataFrame(selected_rows)

    return selected_path, {
        "core_selected": core_selected,
        "secondary_selected": secondary_selected,
        "combined_selected": combined_selected,
    }


def attach_trade_outcomes(selected_path):
    """
    Adds outcomes, features, and conditional ranking metrics to selected path.
    """
    if selected_path.empty:
        return selected_path.copy()

    row_idx = selected_path["row_idx"].to_numpy(dtype=int)
    col_idx = selected_path["selected_col"].to_numpy(dtype=int)

    trades = selected_path.copy()

    trades["candidate"] = MODEL_LABEL

    trades["win"] = win_mat[row_idx, col_idx]
    trades["normalized_pnl_dollars"] = pnl_mat[row_idx, col_idx]
    trades["actual_dte"] = actual_dte_mat[row_idx, col_idx]
    trades["pnl_per_day"] = pnl_per_day_mat[row_idx, col_idx]

    trades["vrp_log"] = vrp_mat[row_idx, col_idx]
    trades["vrp_z_3m"] = z3m_mat[row_idx, col_idx]
    trades["vrp_z_1y"] = z1y_mat[row_idx, col_idx]
    trades["rv21d"] = rv21d_by_date[row_idx]
    trades["rsi14"] = rsi_by_date[row_idx]

    if "spx_close_mat" in globals():
        trades["spx_close"] = spx_close_mat[row_idx, col_idx]
    elif "spx_close_by_date" in globals():
        trades["spx_close"] = np.asarray(spx_close_by_date)[row_idx]
    else:
        trades["spx_close"] = np.nan

    trades = trades.merge(
        tenor_priority_metrics,
        on=["layer", "selected_col", "selected_tenor", "tenor_group"],
        how="left",
    )

    trades = trades.sort_values("date").reset_index(drop=True)

    trades["trade_sequence"] = np.arange(1, len(trades) + 1)
    trades["cum_pnl"] = trades["normalized_pnl_dollars"].cumsum()
    trades["running_max_cum_pnl"] = trades["cum_pnl"].cummax()
    trades["drawdown"] = trades["cum_pnl"] - trades["running_max_cum_pnl"]
    trades["rolling_20_trade_pnl"] = trades["normalized_pnl_dollars"].rolling(20).sum()

    trades["year"] = pd.to_datetime(trades["date"]).dt.year
    trades["month"] = pd.to_datetime(trades["date"]).dt.to_period("M").astype(str)

    return trades


def summarize_trade_df(df, label):
    """
    Summary for one path or subgroup.
    """
    trade_count = int(len(df))

    if trade_count == 0:
        return pd.Series({
            "label": label,
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_pnl_per_day": np.nan,
            "median_pnl_per_day": np.nan,
            "aggregate_pnl_per_day": np.nan,
            "total_pnl": 0.0,
            "total_actual_dte": 0.0,
            "worst_trade_pnl": np.nan,
            "worst_pnl_per_day": np.nan,
            "max_drawdown": np.nan,
            "worst_20_trade_sum": np.nan,
            "avg_selected_tenor": np.nan,
        })

    df = df.sort_values("date").copy()

    df["local_cum_pnl"] = df["normalized_pnl_dollars"].cumsum()
    df["local_running_max"] = df["local_cum_pnl"].cummax()
    df["local_drawdown"] = df["local_cum_pnl"] - df["local_running_max"]
    df["local_rolling_20_trade_pnl"] = df["normalized_pnl_dollars"].rolling(20).sum()

    total_pnl = float(df["normalized_pnl_dollars"].sum())
    total_actual_dte = float(df["actual_dte"].sum())

    return pd.Series({
        "label": label,
        "trade_count": trade_count,
        "trade_frequency": float(trade_count / TRADE_FREQUENCY_DENOMINATOR),
        "win_rate": float(df["win"].mean()),
        "avg_pnl_per_day": float(df["pnl_per_day"].mean()),
        "median_pnl_per_day": float(df["pnl_per_day"].median()),
        "aggregate_pnl_per_day": float(total_pnl / total_actual_dte) if total_actual_dte else np.nan,
        "total_pnl": total_pnl,
        "total_actual_dte": total_actual_dte,
        "worst_trade_pnl": float(df["normalized_pnl_dollars"].min()),
        "worst_pnl_per_day": float(df["pnl_per_day"].min()),
        "max_drawdown": float(df["local_drawdown"].min()),
        "worst_20_trade_sum": float(df["local_rolling_20_trade_pnl"].min()),
        "avg_selected_tenor": float(df["selected_tenor"].mean()),
    })


def summarize_by_group(df, group_cols):
    rows = []

    for keys, sub in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        row = {col: key for col, key in zip(group_cols, keys)}
        summary = summarize_trade_df(sub, label="group").to_dict()
        summary.pop("label", None)

        row.update(summary)
        rows.append(row)

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------
# Build qualification matrices and conditional tenor metrics
# ---------------------------------------------------------------------

core_qualifies_matrix = build_qualifies_matrix(LOCKED_2621_CORE)
secondary_qualifies_matrix = build_qualifies_matrix(LOCKED_2621_SECONDARY)

any_core_qualifies_by_date = core_qualifies_matrix.any(axis=1)
no_core_qualifies_by_date = ~any_core_qualifies_by_date

core_tenor_priority_metrics = build_layer_conditional_tenor_metrics(
    layer_name="Core",
    qualifies_matrix=core_qualifies_matrix,
    date_mask=np.ones(num_sweep_dates, dtype=bool),
)

secondary_tenor_priority_metrics = build_layer_conditional_tenor_metrics(
    layer_name="Secondary",
    qualifies_matrix=secondary_qualifies_matrix,
    date_mask=no_core_qualifies_by_date,
)

tenor_priority_metrics = pd.concat(
    [core_tenor_priority_metrics, secondary_tenor_priority_metrics],
    ignore_index=True,
)

locked_2621_threshold_table = pd.concat(
    [
        build_threshold_table(LOCKED_2621_CORE, "Core"),
        build_threshold_table(LOCKED_2621_SECONDARY, "Secondary"),
    ],
    ignore_index=True,
)

print("\nLocked 2621 qualification counts:")
print("Dates with at least one Core tenor:", int(any_core_qualifies_by_date.sum()))
print("Dates with no Core tenor:", int(no_core_qualifies_by_date.sum()))
print("Dates with at least one Secondary tenor:", int(secondary_qualifies_matrix.any(axis=1).sum()))
print(
    "Dates with at least one Secondary tenor and no Core tenor:",
    int((secondary_qualifies_matrix.any(axis=1) & no_core_qualifies_by_date).sum()),
)

print("\nCore 2621-conditional tenor priority metrics:")
display(
    core_tenor_priority_metrics.sort_values(
        ["conditional_win_probability", "conditional_avg_pnl_per_day"],
        ascending=[False, False],
    )
)

print("\nSecondary 2621-conditional tenor priority metrics:")
display(
    secondary_tenor_priority_metrics.sort_values(
        ["conditional_win_probability", "conditional_avg_pnl_per_day"],
        ascending=[False, False],
    )
)


# ---------------------------------------------------------------------
# Select final model path
# ---------------------------------------------------------------------

selected_path, selected_arrays = select_final_model_path()
locked_2621_selected_trades = attach_trade_outcomes(selected_path)

locked_2621_path_summary = pd.DataFrame(
    [summarize_trade_df(locked_2621_selected_trades, label="Combined")]
)

summary_by_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["layer"],
)

summary_by_year_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["year", "layer"],
)

summary_by_tenor_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["selected_tenor", "layer"],
)

summary_by_group_layer = summarize_by_group(
    locked_2621_selected_trades,
    ["tenor_group", "layer"],
)

worst_trades = (
    locked_2621_selected_trades
    .sort_values("normalized_pnl_dollars", ascending=True)
    .head(20)
    .reset_index(drop=True)
)

worst_20_trade_windows = (
    locked_2621_selected_trades
    .dropna(subset=["rolling_20_trade_pnl"])
    .sort_values("rolling_20_trade_pnl", ascending=True)
    .head(20)
    .reset_index(drop=True)
)

print("\nFinal locked 2621 blended-priority path summary:")
display(locked_2621_path_summary)

print("\nSummary by layer:")
display(summary_by_layer)

print("\nSummary by year and layer:")
display(summary_by_year_layer)

print("\nSummary by tenor and layer:")
display(summary_by_tenor_layer)

print("\nSummary by tenor group and layer:")
display(summary_by_group_layer)

print("\nWorst 20 individual trades:")
display(worst_trades)

print("\nWorst 20-trade rolling windows:")
display(worst_20_trade_windows)


# ---------------------------------------------------------------------
# Save final model outputs
# ---------------------------------------------------------------------

SELECTED_TRADES_CSV_PATH = AUDIT_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.csv"
SELECTED_TRADES_PARQUET_PATH = PROCESSED_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.parquet"

SUMMARY_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_v0_1.csv"
SUMMARY_BY_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_layer_v0_1.csv"
SUMMARY_BY_YEAR_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_year_layer_v0_1.csv"
SUMMARY_BY_TENOR_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_tenor_layer_v0_1.csv"
SUMMARY_BY_GROUP_LAYER_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_by_group_layer_v0_1.csv"

WORST_TRADES_PATH = AUDIT_DIR / f"{MODEL_LABEL}_worst_trades_v0_1.csv"
WORST_WINDOWS_PATH = AUDIT_DIR / f"{MODEL_LABEL}_worst_20_trade_windows_v0_1.csv"

TENOR_PRIORITY_METRICS_PATH = AUDIT_DIR / f"{MODEL_LABEL}_tenor_priority_metrics_v0_1.csv"
THRESHOLD_TABLE_PATH = AUDIT_DIR / f"{MODEL_LABEL}_thresholds_v0_1.csv"
SUMMARY_JSON_PATH = AUDIT_DIR / f"{MODEL_LABEL}_summary_v0_1.json"

locked_2621_selected_trades.to_csv(SELECTED_TRADES_CSV_PATH, index=False)
locked_2621_selected_trades.to_parquet(SELECTED_TRADES_PARQUET_PATH, index=False)

locked_2621_path_summary.to_csv(SUMMARY_PATH, index=False)
summary_by_layer.to_csv(SUMMARY_BY_LAYER_PATH, index=False)
summary_by_year_layer.to_csv(SUMMARY_BY_YEAR_LAYER_PATH, index=False)
summary_by_tenor_layer.to_csv(SUMMARY_BY_TENOR_LAYER_PATH, index=False)
summary_by_group_layer.to_csv(SUMMARY_BY_GROUP_LAYER_PATH, index=False)

worst_trades.to_csv(WORST_TRADES_PATH, index=False)
worst_20_trade_windows.to_csv(WORST_WINDOWS_PATH, index=False)

tenor_priority_metrics.to_csv(TENOR_PRIORITY_METRICS_PATH, index=False)
locked_2621_threshold_table.to_csv(THRESHOLD_TABLE_PATH, index=False)

summary_json = {
    "model_label": MODEL_LABEL,
    "thresholds": "locked_2621",
    "priority_rule": "layer-specific 2621-conditional win probability within 25 bps, then conditional avg P&L/day",
    "win_band": WIN_BAND,
    "trade_count": int(locked_2621_path_summary.iloc[0]["trade_count"]),
    "trade_frequency": float(locked_2621_path_summary.iloc[0]["trade_frequency"]),
    "win_rate": float(locked_2621_path_summary.iloc[0]["win_rate"]),
    "avg_pnl_per_day": float(locked_2621_path_summary.iloc[0]["avg_pnl_per_day"]),
    "aggregate_pnl_per_day": float(locked_2621_path_summary.iloc[0]["aggregate_pnl_per_day"]),
    "total_pnl": float(locked_2621_path_summary.iloc[0]["total_pnl"]),
    "max_drawdown": float(locked_2621_path_summary.iloc[0]["max_drawdown"]),
    "worst_20_trade_sum": float(locked_2621_path_summary.iloc[0]["worst_20_trade_sum"]),
    "avg_selected_tenor": float(locked_2621_path_summary.iloc[0]["avg_selected_tenor"]),
    "selected_trades_csv": str(SELECTED_TRADES_CSV_PATH),
    "selected_trades_parquet": str(SELECTED_TRADES_PARQUET_PATH),
}

with open(SUMMARY_JSON_PATH, "w") as f:
    json.dump(summary_json, f, indent=2)

print("\nSaved final locked 2621 blended-priority outputs:")
print(SELECTED_TRADES_CSV_PATH)
print(SELECTED_TRADES_PARQUET_PATH)
print(SUMMARY_PATH)
print(SUMMARY_BY_LAYER_PATH)
print(SUMMARY_BY_YEAR_LAYER_PATH)
print(SUMMARY_BY_TENOR_LAYER_PATH)
print(SUMMARY_BY_GROUP_LAYER_PATH)
print(WORST_TRADES_PATH)
print(WORST_WINDOWS_PATH)
print(TENOR_PRIORITY_METRICS_PATH)
print(THRESHOLD_TABLE_PATH)
print(SUMMARY_JSON_PATH)

print("\nFinal locked 2621 blended-priority model complete.")

## Future research roadmap

This notebook should remain the clean locked baseline. Future research should be additive and auditable.

### Next: vol-of-vol / regime overlay

The 2026 Feb–Mar weakness should be investigated as a potential regime-control problem, not as an immediate reason to overfit the tenor rule. Candidate overlay ideas to discuss/test later:

- pause new entries when vol-of-vol is elevated;
- tighten Core/Secondary thresholds in high vol-of-vol regimes;
- disable Secondary in stressed regimes;
- reduce sizing instead of changing signal eligibility;
- require stronger VRP/z-score confirmation when VVIX or vol-of-vol is rising.

### After that: sizing

Once the signal + regime overlay is settled, sizing can be layered on top. Initial sizing dimensions:

- Core vs Secondary;
- front / middle / back tenor bucket;
- vol-of-vol or stress regime;
- portfolio-level risk cap / stress-budget usage.

Avoid reintroducing exploratory sweeps or line-fitting into this baseline notebook. Put new research in a separate notebook, then bring only the chosen production candidate back here.


## Vol-of-vol overlay research add-on

This section starts the first regime-overlay test while preserving the locked baseline signal.

### Overlay concept

- Source data: Cboe VVIX daily close.
- VIX denominator: project-built 30D implied volatility from ThetaData-derived `implied_variance`, not official VIX.
- Signal: `log(VVIX / custom_vix30)`.
- Normalization: 6-month rolling z-score, initially 126 trading days.
- First tests:
  - hard filter: no new trade when z-score is above threshold;
  - size reducer: reduce P&L exposure when z-score is above threshold.

This is an overlay on the locked signal, not a replacement for the 2621 thresholds or tenor-priority rule.


In [ ]:
# Cell 8 — Download and store Cboe VVIX daily history
#
# Notes:
#   - This uses daily VVIX close only.
#   - Intraday VVIX implementation is intentionally deferred.
#   - The file is stored in both raw/market and processed form so future
#     notebooks can reuse the same local copy.

import numpy as np
import pandas as pd
from pathlib import Path

VVIX_CBOE_URL = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VVIX_History.csv"

RAW_MARKET_DIR = DATA_DIR / "raw" / "market"
RAW_MARKET_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_VVIX_CSV_PATH = RAW_MARKET_DIR / "vvix_history_cboe_raw.csv"
VVIX_DAILY_CSV_PATH = PROCESSED_DIR / "vvix_daily_v0_1.csv"
VVIX_DAILY_PARQUET_PATH = PROCESSED_DIR / "vvix_daily_v0_1.parquet"

print("Downloading VVIX history from Cboe...")
vvix_raw = pd.read_csv(VVIX_CBOE_URL)
vvix_raw.to_csv(RAW_VVIX_CSV_PATH, index=False)

print("Raw VVIX columns:", list(vvix_raw.columns))
print("Raw VVIX rows:", len(vvix_raw))

# Robust column mapping. Cboe has historically used DATE plus VVIX, but keep
# this tolerant in case column capitalization changes.
vvix_date_col = get_col(
    vvix_raw,
    ["DATE", "Date", "date", "trade_date", "Trade Date"],
    label="VVIX date",
)

vvix_close_col = get_col(
    vvix_raw,
    ["VVIX", "vvix", "CLOSE", "Close", "close", "PX_LAST", "Last", "last"],
    label="VVIX close",
)

vvix_daily = vvix_raw[[vvix_date_col, vvix_close_col]].copy()
vvix_daily.columns = ["date", "vvix_close"]

vvix_daily["date"] = parse_project_date_series(vvix_daily["date"]).dt.normalize()
vvix_daily["vvix_close"] = pd.to_numeric(vvix_daily["vvix_close"], errors="coerce")

vvix_daily = (
    vvix_daily
    .dropna(subset=["date", "vvix_close"])
    .loc[lambda x: x["vvix_close"] > 0]
    .drop_duplicates(subset=["date"], keep="last")
    .sort_values("date")
    .reset_index(drop=True)
)

vvix_daily.to_csv(VVIX_DAILY_CSV_PATH, index=False)
vvix_daily.to_parquet(VVIX_DAILY_PARQUET_PATH, index=False)

print("\nStored VVIX daily history:")
print("  Raw path:      ", RAW_VVIX_CSV_PATH)
print("  Processed CSV: ", VVIX_DAILY_CSV_PATH)
print("  Processed PQ:  ", VVIX_DAILY_PARQUET_PATH)
print("  Rows:          ", len(vvix_daily))
print("  Date range:    ", vvix_daily["date"].min(), "to", vvix_daily["date"].max())

print("\nVVIX daily tail:")
display(vvix_daily.tail(10))


In [ ]:
# Cell 9 — Build 6-month z-score of log(VVIX / custom VIX30)
#
# Denominator:
#   custom_vix30 = sqrt(30D implied variance) * 100
#
# Why custom VIX30:
#   This keeps the overlay aligned with the project surface built from ThetaData,
#   instead of mixing in official VIX methodology.
#
# Lookahead convention:
#   By default, the rolling mean/std include the current observation.
#   This is consistent with an end-of-day signal using same-day close inputs.
#   If execution is moved earlier than the close, set
#   VOV_SHIFT_ROLLING_STATS_BY_ONE = True.

import numpy as np
import pandas as pd
from pathlib import Path

VOV_ROLLING_WINDOW = 126       # approximately 6 trading months
VOV_MIN_PERIODS = 126
VOV_SHIFT_ROLLING_STATS_BY_ONE = False

VOV_FEATURE_CSV_PATH = PROCESSED_DIR / "vol_of_vol_overlay_features_v0_1.csv"
VOV_FEATURE_PARQUET_PATH = PROCESSED_DIR / "vol_of_vol_overlay_features_v0_1.parquet"

if "features" in globals():
    features_for_vov = features.copy()
else:
    features_for_vov = pd.read_parquet(FEATURE_PANEL_PATH)
    features_for_vov["date"] = pd.to_datetime(features_for_vov["date"]).dt.normalize()

if "vvix_daily" not in globals():
    vvix_daily = pd.read_parquet(VVIX_DAILY_PARQUET_PATH)
    vvix_daily["date"] = pd.to_datetime(vvix_daily["date"]).dt.normalize()

require_columns(features_for_vov, ["date", "tenor", "implied_variance"], label="feature panel for VOV overlay")

vix30_daily = (
    features_for_vov
    .loc[features_for_vov["tenor"].astype(int).eq(30), ["date", "implied_variance"]]
    .copy()
)

vix30_daily["date"] = pd.to_datetime(vix30_daily["date"]).dt.normalize()
vix30_daily["implied_variance"] = pd.to_numeric(vix30_daily["implied_variance"], errors="coerce")
vix30_daily = vix30_daily.dropna(subset=["date", "implied_variance"])
vix30_daily = vix30_daily[vix30_daily["implied_variance"] > 0].copy()

# Most project implied-variance series are annualized decimal variance.
# In that case sqrt(var) is decimal volatility, so multiply by 100 to put it
# on the same percentage-point scale as VVIX.
vix30_daily["custom_vix30_raw_sqrt"] = np.sqrt(vix30_daily["implied_variance"])
raw_sqrt_median = float(vix30_daily["custom_vix30_raw_sqrt"].median())

if raw_sqrt_median < 3.0:
    vix_scale = 100.0
else:
    vix_scale = 1.0

vix30_daily["custom_vix30"] = vix30_daily["custom_vix30_raw_sqrt"] * vix_scale

vix30_daily = (
    vix30_daily
    .drop_duplicates(subset=["date"], keep="last")
    .sort_values("date")
    .reset_index(drop=True)
)

vov_features = (
    vix30_daily[["date", "implied_variance", "custom_vix30"]]
    .merge(vvix_daily[["date", "vvix_close"]], on="date", how="left")
    .sort_values("date")
    .reset_index(drop=True)
)

vov_features["vvix_vix_ratio"] = vov_features["vvix_close"] / vov_features["custom_vix30"]
vov_features["log_vvix_vix_ratio"] = np.log(vov_features["vvix_vix_ratio"])

ratio = vov_features["log_vvix_vix_ratio"]
rolling_mean = ratio.rolling(VOV_ROLLING_WINDOW, min_periods=VOV_MIN_PERIODS).mean()
rolling_std = ratio.rolling(VOV_ROLLING_WINDOW, min_periods=VOV_MIN_PERIODS).std(ddof=0)

if VOV_SHIFT_ROLLING_STATS_BY_ONE:
    rolling_mean = rolling_mean.shift(1)
    rolling_std = rolling_std.shift(1)

vov_features["log_vvix_vix_ratio_mean_6m"] = rolling_mean
vov_features["log_vvix_vix_ratio_std_6m"] = rolling_std

with np.errstate(divide="ignore", invalid="ignore"):
    vov_features["log_vvix_vix_ratio_z_6m"] = (
        (ratio - rolling_mean) / rolling_std
    )

vov_features.loc[
    ~np.isfinite(vov_features["log_vvix_vix_ratio_z_6m"]),
    "log_vvix_vix_ratio_z_6m",
] = np.nan

vov_features["vov_rolling_window_days"] = VOV_ROLLING_WINDOW
vov_features["vov_min_periods"] = VOV_MIN_PERIODS
vov_features["vov_shift_rolling_stats_by_one"] = VOV_SHIFT_ROLLING_STATS_BY_ONE

vov_features.to_csv(VOV_FEATURE_CSV_PATH, index=False)
vov_features.to_parquet(VOV_FEATURE_PARQUET_PATH, index=False)

print("Built vol-of-vol overlay features")
print("  VIX scale applied to sqrt(implied_variance):", vix_scale)
print("  Median raw sqrt implied variance:", raw_sqrt_median)
print("  Rows:", len(vov_features))
print("  Date range:", vov_features["date"].min(), "to", vov_features["date"].max())
print("  Non-null VVIX rows:", int(vov_features["vvix_close"].notna().sum()))
print("  Non-null z-score rows:", int(vov_features["log_vvix_vix_ratio_z_6m"].notna().sum()))
print("  Saved CSV:", VOV_FEATURE_CSV_PATH)
print("  Saved PQ: ", VOV_FEATURE_PARQUET_PATH)

print("\nOverlay feature tail:")
display(vov_features.tail(10))

print("\nOverlay feature summary:")
display(
    vov_features[
        [
            "custom_vix30",
            "vvix_close",
            "vvix_vix_ratio",
            "log_vvix_vix_ratio",
            "log_vvix_vix_ratio_z_6m",
        ]
    ].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)


In [ ]:
# Cell 10 — First-pass vol-of-vol overlay tests on locked 2621 selected trades
#
# Tests:
#   1. Hard filter:
#        if log(VVIX/custom VIX30) 6m z-score > threshold, skip new trade.
#
#   2. Simple size reducer:
#        if z-score > threshold, keep trade but reduce P&L exposure to 50%.
#
# This is intentionally simple. Later we can test tiered sizing schedules and
# condition overlays by Core/Secondary and tenor bucket.

import numpy as np
import pandas as pd
from pathlib import Path

VOV_Z_COL = "log_vvix_vix_ratio_z_6m"
VOV_THRESHOLDS = [1.00, 1.25, 1.50, 1.75, 2.00]
REDUCED_SIZE_MULTIPLIER = 0.50

VOV_OVERLAY_SUMMARY_PATH = AUDIT_DIR / "vol_of_vol_overlay_log_ratio_6m_z_bakeoff_v0_1.csv"
VOV_OVERLAY_TRADES_PATH = AUDIT_DIR / "vol_of_vol_overlay_log_ratio_6m_z_trade_detail_v0_1.csv"
VOV_OVERLAY_BLOCKED_PATH = AUDIT_DIR / "vol_of_vol_overlay_log_ratio_6m_z_blocked_trades_v0_1.csv"

if "locked_2621_selected_trades" in globals():
    base_trades = locked_2621_selected_trades.copy()
else:
    BASE_SELECTED_TRADES_PATH = PROCESSED_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.parquet"
    if not BASE_SELECTED_TRADES_PATH.exists():
        raise FileNotFoundError(
            f"Missing selected trades file: {BASE_SELECTED_TRADES_PATH}. "
            "Run the locked 2621 model cell first."
        )
    base_trades = pd.read_parquet(BASE_SELECTED_TRADES_PATH)

if "vov_features" not in globals():
    vov_features = pd.read_parquet(VOV_FEATURE_PARQUET_PATH)

base_trades = base_trades.copy()
base_trades["date"] = pd.to_datetime(base_trades["date"]).dt.normalize()
vov_features = vov_features.copy()
vov_features["date"] = pd.to_datetime(vov_features["date"]).dt.normalize()

trades_with_vov = base_trades.merge(
    vov_features[
        [
            "date",
            "custom_vix30",
            "vvix_close",
            "vvix_vix_ratio",
            "log_vvix_vix_ratio",
            "log_vvix_vix_ratio_z_6m",
        ]
    ],
    on="date",
    how="left",
)

missing_overlay = trades_with_vov[VOV_Z_COL].isna().sum()
print("Selected trades:", len(trades_with_vov))
print("Trades missing VOV z-score:", int(missing_overlay))

if missing_overlay > 0:
    print("\nMissing VOV z-score dates:")
    display(
        trades_with_vov.loc[trades_with_vov[VOV_Z_COL].isna(), ["date", "layer", "selected_tenor"]]
        .head(20)
    )

# For first pass, require overlay data. If this drops trades, that should be
# visible and intentional.
trades_with_vov = trades_with_vov.dropna(subset=[VOV_Z_COL]).copy()
trades_with_vov = trades_with_vov.sort_values("date").reset_index(drop=True)
trades_with_vov["year"] = pd.to_datetime(trades_with_vov["date"]).dt.year
trades_with_vov["month"] = pd.to_datetime(trades_with_vov["date"]).dt.to_period("M").astype(str)


def summarize_effective_pnl(df, label, effective_pnl_col="effective_pnl", size_col="size_multiplier"):
    df = df.copy().sort_values("date")
    trade_count = int(len(df))

    if trade_count == 0:
        return {
            "label": label,
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_size_multiplier": np.nan,
            "notional_equiv_trade_count": 0.0,
            "avg_effective_pnl_per_day": np.nan,
            "aggregate_effective_pnl_per_day": np.nan,
            "total_effective_pnl": 0.0,
            "max_drawdown": np.nan,
            "worst_20_trade_sum": np.nan,
            "avg_selected_tenor": np.nan,
            "avg_vov_z": np.nan,
            "max_vov_z": np.nan,
        }

    df[effective_pnl_col] = pd.to_numeric(df[effective_pnl_col], errors="coerce")
    df["actual_dte"] = pd.to_numeric(df["actual_dte"], errors="coerce")

    df["effective_pnl_per_day"] = df[effective_pnl_col] / df["actual_dte"]
    df["effective_cum_pnl"] = df[effective_pnl_col].cumsum()
    df["effective_running_max"] = df["effective_cum_pnl"].cummax()
    df["effective_drawdown"] = df["effective_cum_pnl"] - df["effective_running_max"]
    df["effective_rolling_20_trade_pnl"] = df[effective_pnl_col].rolling(20).sum()

    total_effective_pnl = float(df[effective_pnl_col].sum())
    total_actual_dte = float(df["actual_dte"].sum())

    return {
        "label": label,
        "trade_count": trade_count,
        "trade_frequency": float(trade_count / TRADE_FREQUENCY_DENOMINATOR),
        "win_rate": float(df["win"].mean()) if "win" in df.columns else np.nan,
        "avg_size_multiplier": float(df[size_col].mean()) if size_col in df.columns else 1.0,
        "notional_equiv_trade_count": float(df[size_col].sum()) if size_col in df.columns else float(trade_count),
        "avg_effective_pnl_per_day": float(df["effective_pnl_per_day"].mean()),
        "aggregate_effective_pnl_per_day": float(total_effective_pnl / total_actual_dte) if total_actual_dte else np.nan,
        "total_effective_pnl": total_effective_pnl,
        "max_drawdown": float(df["effective_drawdown"].min()),
        "worst_20_trade_sum": float(df["effective_rolling_20_trade_pnl"].min()),
        "avg_selected_tenor": float(df["selected_tenor"].mean()),
        "avg_vov_z": float(df[VOV_Z_COL].mean()),
        "max_vov_z": float(df[VOV_Z_COL].max()),
    }


def add_sample_summaries(rows, scenario_name, action, threshold, df):
    samples = {
        "full_sample": df,
        "core": df[df["layer"].eq("Core")],
        "secondary": df[df["layer"].eq("Secondary")],
        "2026_all": df[df["year"].eq(2026)],
        "2026_core": df[df["year"].eq(2026) & df["layer"].eq("Core")],
    }

    for sample_name, sample_df in samples.items():
        row = summarize_effective_pnl(sample_df, label=sample_name)
        row.update({
            "scenario": scenario_name,
            "action": action,
            "threshold": threshold,
            "sample": sample_name,
        })
        rows.append(row)


summary_rows = []
trade_detail_frames = []
blocked_frames = []

# Baseline: no overlay.
baseline = trades_with_vov.copy()
baseline["scenario"] = "baseline_no_overlay"
baseline["action"] = "none"
baseline["threshold"] = np.nan
baseline["blocked_by_overlay"] = False
baseline["size_multiplier"] = 1.0
baseline["effective_pnl"] = baseline["normalized_pnl_dollars"]

add_sample_summaries(
    rows=summary_rows,
    scenario_name="baseline_no_overlay",
    action="none",
    threshold=np.nan,
    df=baseline,
)
trade_detail_frames.append(baseline)

for threshold in VOV_THRESHOLDS:
    # Hard filter / no trade.
    hard = trades_with_vov.copy()
    hard["scenario"] = f"hard_filter_z_gt_{threshold:.2f}"
    hard["action"] = "no_trade_if_z_above_threshold"
    hard["threshold"] = threshold
    hard["blocked_by_overlay"] = hard[VOV_Z_COL] > threshold
    hard["size_multiplier"] = np.where(hard["blocked_by_overlay"], 0.0, 1.0)
    hard["effective_pnl"] = hard["normalized_pnl_dollars"] * hard["size_multiplier"]

    blocked = hard[hard["blocked_by_overlay"]].copy()
    if not blocked.empty:
        blocked_frames.append(blocked)

    hard_kept = hard[~hard["blocked_by_overlay"]].copy()

    add_sample_summaries(
        rows=summary_rows,
        scenario_name=f"hard_filter_z_gt_{threshold:.2f}",
        action="no_trade_if_z_above_threshold",
        threshold=threshold,
        df=hard_kept,
    )
    trade_detail_frames.append(hard)

    # Simple reduced-size overlay.
    reduced = trades_with_vov.copy()
    reduced["scenario"] = f"reduce_50pct_if_z_gt_{threshold:.2f}"
    reduced["action"] = "reduce_size_if_z_above_threshold"
    reduced["threshold"] = threshold
    reduced["blocked_by_overlay"] = False
    reduced["size_multiplier"] = np.where(reduced[VOV_Z_COL] > threshold, REDUCED_SIZE_MULTIPLIER, 1.0)
    reduced["effective_pnl"] = reduced["normalized_pnl_dollars"] * reduced["size_multiplier"]

    add_sample_summaries(
        rows=summary_rows,
        scenario_name=f"reduce_50pct_if_z_gt_{threshold:.2f}",
        action="reduce_size_if_z_above_threshold",
        threshold=threshold,
        df=reduced,
    )
    trade_detail_frames.append(reduced)

vov_overlay_summary = pd.DataFrame(summary_rows)
vov_overlay_trade_detail = pd.concat(trade_detail_frames, ignore_index=True)

if blocked_frames:
    vov_overlay_blocked_trades = pd.concat(blocked_frames, ignore_index=True)
else:
    vov_overlay_blocked_trades = pd.DataFrame()

# Add deltas versus baseline within each sample.
baseline_by_sample = (
    vov_overlay_summary
    .loc[vov_overlay_summary["scenario"].eq("baseline_no_overlay")]
    .set_index("sample")
)

for metric in [
    "trade_count",
    "trade_frequency",
    "win_rate",
    "notional_equiv_trade_count",
    "avg_effective_pnl_per_day",
    "aggregate_effective_pnl_per_day",
    "total_effective_pnl",
    "max_drawdown",
    "worst_20_trade_sum",
]:
    vov_overlay_summary[f"{metric}_minus_baseline"] = vov_overlay_summary.apply(
        lambda row: row[metric] - baseline_by_sample.loc[row["sample"], metric]
        if row["sample"] in baseline_by_sample.index else np.nan,
        axis=1,
    )

vov_overlay_summary = vov_overlay_summary.sort_values(
    ["sample", "action", "threshold"],
    ascending=[True, True, True],
).reset_index(drop=True)

vov_overlay_summary.to_csv(VOV_OVERLAY_SUMMARY_PATH, index=False)
vov_overlay_trade_detail.to_csv(VOV_OVERLAY_TRADES_PATH, index=False)

if not vov_overlay_blocked_trades.empty:
    vov_overlay_blocked_trades.to_csv(VOV_OVERLAY_BLOCKED_PATH, index=False)

print("Vol-of-vol overlay bakeoff complete")
print("  Summary path:", VOV_OVERLAY_SUMMARY_PATH)
print("  Trade detail path:", VOV_OVERLAY_TRADES_PATH)
if not vov_overlay_blocked_trades.empty:
    print("  Blocked trades path:", VOV_OVERLAY_BLOCKED_PATH)

print("\nFull-sample overlay summary:")
display(
    vov_overlay_summary
    .loc[vov_overlay_summary["sample"].eq("full_sample")]
    .sort_values(["action", "threshold"])
)

print("\n2026 Core overlay summary:")
display(
    vov_overlay_summary
    .loc[vov_overlay_summary["sample"].eq("2026_core")]
    .sort_values(["action", "threshold"])
)

print("\nWorst blocked trades by threshold/action:")
if not vov_overlay_blocked_trades.empty:
    display(
        vov_overlay_blocked_trades
        .sort_values("normalized_pnl_dollars", ascending=True)
        [[
            "scenario",
            "threshold",
            "date",
            "layer",
            "selected_tenor",
            "tenor_group",
            "win",
            "normalized_pnl_dollars",
            "pnl_per_day",
            "custom_vix30",
            "vvix_close",
            "vvix_vix_ratio",
            VOV_Z_COL,
        ]]
        .head(30)
    )
else:
    print("No trades were blocked at tested thresholds.")


## Vol-of-vol overlay research — second pass

The first-pass `log(VVIX / custom VIX30)` z-score test was useful, but it did not identify the Feb–Mar 2026 Core drawdown cluster. This second-pass section keeps the same locked 2621 selected-trade path and tests broader regime candidates:

- custom VIX30 5-day and 10-day acceleration
- custom VIX30 realized vol-of-vol
- VVIX alone
- VVIX residual relative to custom VIX30

The goal is not to change the signal yet. The goal is to identify whether any overlay variable would have flagged hostile short-vol regimes before or during bad clusters without deleting too much long-run edge.


In [ ]:
# Cell 11 — Build second-pass vol-of-vol regime features
#
# Adds overlay candidates beyond the first-pass log(VVIX / custom VIX30) ratio:
#   1. VIX30 acceleration:
#        - 5d change in custom VIX30
#        - 10d change in custom VIX30
#        - 5d log change in custom VIX30
#        - 10d log change in custom VIX30
#
#   2. Realized vol-of-vol:
#        - 10d rolling stdev of daily custom VIX30 changes
#        - 21d rolling stdev of daily custom VIX30 changes
#
#   3. VVIX alone:
#        - 6m z-score of log(VVIX)
#
#   4. Residual VVIX:
#        - rolling residual of log(VVIX) after regressing it on log(custom VIX30)
#        - standardized by the rolling residual stdev
#
# Convention:
#   Rolling z-scores use a 126 trading-day lookback as the starting 6m window.
#   By default, the current observation is included, consistent with an EOD signal.
#   If execution is moved before the close, set SECOND_PASS_SHIFT_ROLLING_STATS_BY_ONE = True.

import numpy as np
import pandas as pd
from pathlib import Path

SECOND_PASS_WINDOW = 126
SECOND_PASS_MIN_PERIODS = 126
SECOND_PASS_SHIFT_ROLLING_STATS_BY_ONE = False

VOV_SECOND_PASS_FEATURE_CSV_PATH = PROCESSED_DIR / "vol_of_vol_overlay_second_pass_features_v0_1.csv"
VOV_SECOND_PASS_FEATURE_PARQUET_PATH = PROCESSED_DIR / "vol_of_vol_overlay_second_pass_features_v0_1.parquet"

if "vov_features" not in globals():
    vov_features = pd.read_parquet(VOV_FEATURE_PARQUET_PATH)

vov_second = vov_features.copy()
vov_second["date"] = pd.to_datetime(vov_second["date"]).dt.normalize()
vov_second = vov_second.sort_values("date").reset_index(drop=True)

required_vov_cols = ["date", "custom_vix30", "vvix_close"]
missing_vov_cols = [c for c in required_vov_cols if c not in vov_second.columns]
if missing_vov_cols:
    raise ValueError(f"vov_features missing required columns: {missing_vov_cols}")

vov_second["custom_vix30"] = pd.to_numeric(vov_second["custom_vix30"], errors="coerce")
vov_second["vvix_close"] = pd.to_numeric(vov_second["vvix_close"], errors="coerce")

vov_second.loc[vov_second["custom_vix30"] <= 0, "custom_vix30"] = np.nan
vov_second.loc[vov_second["vvix_close"] <= 0, "vvix_close"] = np.nan

vov_second["log_custom_vix30"] = np.log(vov_second["custom_vix30"])
vov_second["log_vvix"] = np.log(vov_second["vvix_close"])


def rolling_zscore(series, window=SECOND_PASS_WINDOW, min_periods=SECOND_PASS_MIN_PERIODS, shift_stats=SECOND_PASS_SHIFT_ROLLING_STATS_BY_ONE):
    series = pd.to_numeric(series, errors="coerce")
    rolling_mean = series.rolling(window, min_periods=min_periods).mean()
    rolling_std = series.rolling(window, min_periods=min_periods).std(ddof=0)

    if shift_stats:
        rolling_mean = rolling_mean.shift(1)
        rolling_std = rolling_std.shift(1)

    with np.errstate(divide="ignore", invalid="ignore"):
        z = (series - rolling_mean) / rolling_std

    z = z.where(np.isfinite(z), np.nan)
    return z, rolling_mean, rolling_std


# ---------------------------------------------------------------------
# VIX30 acceleration features
# ---------------------------------------------------------------------

vov_second["custom_vix30_change_1d"] = vov_second["custom_vix30"].diff(1)
vov_second["custom_vix30_change_5d"] = vov_second["custom_vix30"].diff(5)
vov_second["custom_vix30_change_10d"] = vov_second["custom_vix30"].diff(10)

vov_second["log_custom_vix30_change_1d"] = vov_second["log_custom_vix30"].diff(1)
vov_second["log_custom_vix30_change_5d"] = vov_second["log_custom_vix30"].diff(5)
vov_second["log_custom_vix30_change_10d"] = vov_second["log_custom_vix30"].diff(10)

for col in [
    "custom_vix30_change_5d",
    "custom_vix30_change_10d",
    "log_custom_vix30_change_5d",
    "log_custom_vix30_change_10d",
]:
    z, mean, std = rolling_zscore(vov_second[col])
    vov_second[f"{col}_z_6m"] = z
    vov_second[f"{col}_mean_6m"] = mean
    vov_second[f"{col}_std_6m"] = std


# ---------------------------------------------------------------------
# Realized vol-of-vol features
# ---------------------------------------------------------------------

# Units: custom VIX30 points per day.
vov_second["custom_vix30_rvov_10d"] = (
    vov_second["custom_vix30_change_1d"]
    .rolling(10, min_periods=10)
    .std(ddof=0)
)

vov_second["custom_vix30_rvov_21d"] = (
    vov_second["custom_vix30_change_1d"]
    .rolling(21, min_periods=21)
    .std(ddof=0)
)

# Log units: daily log-change volatility of custom VIX30.
vov_second["log_custom_vix30_rvov_10d"] = (
    vov_second["log_custom_vix30_change_1d"]
    .rolling(10, min_periods=10)
    .std(ddof=0)
)

vov_second["log_custom_vix30_rvov_21d"] = (
    vov_second["log_custom_vix30_change_1d"]
    .rolling(21, min_periods=21)
    .std(ddof=0)
)

for col in [
    "custom_vix30_rvov_10d",
    "custom_vix30_rvov_21d",
    "log_custom_vix30_rvov_10d",
    "log_custom_vix30_rvov_21d",
]:
    z, mean, std = rolling_zscore(vov_second[col])
    vov_second[f"{col}_z_6m"] = z
    vov_second[f"{col}_mean_6m"] = mean
    vov_second[f"{col}_std_6m"] = std


# ---------------------------------------------------------------------
# VVIX-alone feature
# ---------------------------------------------------------------------

z, mean, std = rolling_zscore(vov_second["log_vvix"])
vov_second["log_vvix_z_6m"] = z
vov_second["log_vvix_mean_6m"] = mean
vov_second["log_vvix_std_6m"] = std


# ---------------------------------------------------------------------
# Residual VVIX feature
# ---------------------------------------------------------------------

def rolling_residual_y_on_x(y, x, window=SECOND_PASS_WINDOW, min_periods=SECOND_PASS_MIN_PERIODS, shift_window=SECOND_PASS_SHIFT_ROLLING_STATS_BY_ONE):
    """
    Rolling residual from y ~ alpha + beta * x.

    If shift_window=False:
        Uses the window ending at the current observation.
        This is consistent with EOD signal construction.

    If shift_window=True:
        Uses the prior window only.
        This is more conservative for pre-close execution.
    """
    y = pd.to_numeric(y, errors="coerce").to_numpy(dtype=float)
    x = pd.to_numeric(x, errors="coerce").to_numpy(dtype=float)

    n = len(y)
    residual = np.full(n, np.nan)
    residual_z = np.full(n, np.nan)
    alpha_arr = np.full(n, np.nan)
    beta_arr = np.full(n, np.nan)
    resid_std_arr = np.full(n, np.nan)

    for i in range(n):
        if not np.isfinite(y[i]) or not np.isfinite(x[i]):
            continue

        end_idx = i - 1 if shift_window else i
        if end_idx < 0:
            continue

        start_idx = max(0, end_idx - window + 1)
        idx = np.arange(start_idx, end_idx + 1)

        mask = np.isfinite(y[idx]) & np.isfinite(x[idx])
        idx = idx[mask]

        if len(idx) < min_periods:
            continue

        xw = x[idx]
        yw = y[idx]

        x_mean = float(np.mean(xw))
        y_mean = float(np.mean(yw))
        x_var = float(np.mean((xw - x_mean) ** 2))

        if x_var <= 0 or not np.isfinite(x_var):
            continue

        beta = float(np.mean((xw - x_mean) * (yw - y_mean)) / x_var)
        alpha = float(y_mean - beta * x_mean)

        fitted_w = alpha + beta * xw
        resid_w = yw - fitted_w
        resid_std = float(np.std(resid_w, ddof=0))

        current_residual = float(y[i] - (alpha + beta * x[i]))

        alpha_arr[i] = alpha
        beta_arr[i] = beta
        residual[i] = current_residual
        resid_std_arr[i] = resid_std

        if resid_std > 0 and np.isfinite(resid_std):
            residual_z[i] = current_residual / resid_std

    return pd.DataFrame({
        "rolling_alpha": alpha_arr,
        "rolling_beta": beta_arr,
        "residual": residual,
        "residual_std": resid_std_arr,
        "residual_z": residual_z,
    })


resid_df = rolling_residual_y_on_x(
    y=vov_second["log_vvix"],
    x=vov_second["log_custom_vix30"],
)

vov_second["log_vvix_residual_vs_log_vix30"] = resid_df["residual"]
vov_second["log_vvix_residual_vs_log_vix30_z_6m"] = resid_df["residual_z"]
vov_second["log_vvix_vs_log_vix30_rolling_alpha_6m"] = resid_df["rolling_alpha"]
vov_second["log_vvix_vs_log_vix30_rolling_beta_6m"] = resid_df["rolling_beta"]
vov_second["log_vvix_vs_log_vix30_residual_std_6m"] = resid_df["residual_std"]

vov_second["second_pass_rolling_window_days"] = SECOND_PASS_WINDOW
vov_second["second_pass_min_periods"] = SECOND_PASS_MIN_PERIODS
vov_second["second_pass_shift_rolling_stats_by_one"] = SECOND_PASS_SHIFT_ROLLING_STATS_BY_ONE

vov_second.to_csv(VOV_SECOND_PASS_FEATURE_CSV_PATH, index=False)
vov_second.to_parquet(VOV_SECOND_PASS_FEATURE_PARQUET_PATH, index=False)

SECOND_PASS_SIGNAL_COLS = [
    "custom_vix30_change_5d_z_6m",
    "custom_vix30_change_10d_z_6m",
    "log_custom_vix30_change_5d_z_6m",
    "log_custom_vix30_change_10d_z_6m",
    "custom_vix30_rvov_10d_z_6m",
    "custom_vix30_rvov_21d_z_6m",
    "log_custom_vix30_rvov_10d_z_6m",
    "log_custom_vix30_rvov_21d_z_6m",
    "log_vvix_z_6m",
    "log_vvix_residual_vs_log_vix30_z_6m",
]

print("Built second-pass vol-of-vol overlay features")
print("  Rows:", len(vov_second))
print("  Date range:", vov_second["date"].min(), "to", vov_second["date"].max())
print("  Saved CSV:", VOV_SECOND_PASS_FEATURE_CSV_PATH)
print("  Saved PQ: ", VOV_SECOND_PASS_FEATURE_PARQUET_PATH)

print("\nSecond-pass signal availability:")
availability = []
for col in SECOND_PASS_SIGNAL_COLS:
    availability.append({
        "signal_col": col,
        "non_null_rows": int(vov_second[col].notna().sum()),
        "first_non_null_date": vov_second.loc[vov_second[col].notna(), "date"].min() if vov_second[col].notna().any() else pd.NaT,
        "last_non_null_date": vov_second.loc[vov_second[col].notna(), "date"].max() if vov_second[col].notna().any() else pd.NaT,
        "mean": float(vov_second[col].mean()),
        "std": float(vov_second[col].std(ddof=0)),
        "p95": float(vov_second[col].quantile(0.95)),
        "p99": float(vov_second[col].quantile(0.99)),
        "max": float(vov_second[col].max()),
    })

display(pd.DataFrame(availability))

print("\nSecond-pass feature tail:")
display(
    vov_second[
        [
            "date",
            "custom_vix30",
            "vvix_close",
            "custom_vix30_change_5d_z_6m",
            "custom_vix30_change_10d_z_6m",
            "custom_vix30_rvov_10d_z_6m",
            "custom_vix30_rvov_21d_z_6m",
            "log_vvix_z_6m",
            "log_vvix_residual_vs_log_vix30_z_6m",
        ]
    ].tail(15)
)


In [ ]:
# Cell 12 — Second-pass vol-of-vol overlay bakeoff
#
# Tests each second-pass signal as a simple overlay on the locked 2621 selected trades.
#
# For each signal and threshold:
#   1. Hard filter:
#        no new trade if signal z-score > threshold
#
#   2. Size reducer:
#        trade still occurs, but size is reduced to 50% if signal z-score > threshold
#
# Samples reported:
#   - full_sample
#   - Core
#   - Secondary
#   - 2026_all
#   - 2026_core
#   - 2026_Q1_core
#   - 2026_Feb_Mar_core
#
# The Feb/Mar sample is included because the prior diagnostics showed that this
# was the weak 2026 cluster we care about most.

import numpy as np
import pandas as pd
from pathlib import Path

SECOND_PASS_THRESHOLDS = [1.00, 1.25, 1.50, 1.75, 2.00]
SECOND_PASS_REDUCED_SIZE_MULTIPLIER = 0.50

VOV_SECOND_PASS_SUMMARY_PATH = AUDIT_DIR / "vol_of_vol_overlay_second_pass_bakeoff_v0_1.csv"
VOV_SECOND_PASS_TRADES_PATH = AUDIT_DIR / "vol_of_vol_overlay_second_pass_trade_detail_v0_1.csv"
VOV_SECOND_PASS_HIT_TRADES_PATH = AUDIT_DIR / "vol_of_vol_overlay_second_pass_hit_trades_v0_1.csv"

if "locked_2621_selected_trades" in globals():
    base_trades = locked_2621_selected_trades.copy()
else:
    BASE_SELECTED_TRADES_PATH = PROCESSED_DIR / f"{MODEL_LABEL}_selected_trades_v0_1.parquet"
    if not BASE_SELECTED_TRADES_PATH.exists():
        raise FileNotFoundError(
            f"Missing selected trades file: {BASE_SELECTED_TRADES_PATH}. "
            "Run the locked 2621 model cell first."
        )
    base_trades = pd.read_parquet(BASE_SELECTED_TRADES_PATH)

if "vov_second" not in globals():
    vov_second = pd.read_parquet(VOV_SECOND_PASS_FEATURE_PARQUET_PATH)

base_trades = base_trades.copy()
base_trades["date"] = pd.to_datetime(base_trades["date"]).dt.normalize()
base_trades["year"] = base_trades["date"].dt.year
base_trades["month"] = base_trades["date"].dt.to_period("M").astype(str)

vov_second = vov_second.copy()
vov_second["date"] = pd.to_datetime(vov_second["date"]).dt.normalize()

if "SECOND_PASS_SIGNAL_COLS" not in globals():
    SECOND_PASS_SIGNAL_COLS = [
        "custom_vix30_change_5d_z_6m",
        "custom_vix30_change_10d_z_6m",
        "log_custom_vix30_change_5d_z_6m",
        "log_custom_vix30_change_10d_z_6m",
        "custom_vix30_rvov_10d_z_6m",
        "custom_vix30_rvov_21d_z_6m",
        "log_custom_vix30_rvov_10d_z_6m",
        "log_custom_vix30_rvov_21d_z_6m",
        "log_vvix_z_6m",
        "log_vvix_residual_vs_log_vix30_z_6m",
    ]

missing_signal_cols = [c for c in SECOND_PASS_SIGNAL_COLS if c not in vov_second.columns]
if missing_signal_cols:
    raise ValueError(f"Missing second-pass signal columns. Run Cell 11 first: {missing_signal_cols}")

feature_cols_for_merge = [
    "date",
    "custom_vix30",
    "vvix_close",
    "log_custom_vix30",
    "log_vvix",
] + SECOND_PASS_SIGNAL_COLS

trades_base_second = base_trades.merge(
    vov_second[feature_cols_for_merge],
    on="date",
    how="left",
)

print("Selected trades:", len(trades_base_second))
print("Second-pass signal missing counts on selected trades:")
display(
    pd.DataFrame([
        {
            "signal_col": col,
            "missing_trades": int(trades_base_second[col].isna().sum()),
            "non_missing_trades": int(trades_base_second[col].notna().sum()),
        }
        for col in SECOND_PASS_SIGNAL_COLS
    ])
)


def add_sample_flags(df):
    df = df.copy()
    df["is_core"] = df["layer"].eq("Core")
    df["is_secondary"] = df["layer"].eq("Secondary")
    df["is_2026"] = df["date"].dt.year.eq(2026)
    df["is_2026_q1"] = df["date"].between(pd.Timestamp("2026-01-01"), pd.Timestamp("2026-03-31"))
    df["is_2026_feb_mar"] = df["date"].between(pd.Timestamp("2026-02-01"), pd.Timestamp("2026-03-31"))
    return df


trades_base_second = add_sample_flags(trades_base_second)


def summarize_effective_pnl_v2(df, label, signal_col=None, effective_pnl_col="effective_pnl", size_col="size_multiplier"):
    df = df.copy().sort_values("date")
    trade_count = int(len(df))

    if trade_count == 0:
        return {
            "label": label,
            "trade_count": 0,
            "trade_frequency": 0.0,
            "win_rate": np.nan,
            "avg_size_multiplier": np.nan,
            "notional_equiv_trade_count": 0.0,
            "avg_effective_pnl_per_day": np.nan,
            "aggregate_effective_pnl_per_day": np.nan,
            "total_effective_pnl": 0.0,
            "max_drawdown": np.nan,
            "worst_20_trade_sum": np.nan,
            "avg_selected_tenor": np.nan,
            "avg_signal_z": np.nan,
            "max_signal_z": np.nan,
            "min_signal_z": np.nan,
            "overlay_hit_count": 0,
            "overlay_hit_rate": np.nan,
        }

    df[effective_pnl_col] = pd.to_numeric(df[effective_pnl_col], errors="coerce")
    df["actual_dte"] = pd.to_numeric(df["actual_dte"], errors="coerce")

    df["effective_pnl_per_day"] = df[effective_pnl_col] / df["actual_dte"]
    df["effective_cum_pnl"] = df[effective_pnl_col].cumsum()
    df["effective_running_max"] = df["effective_cum_pnl"].cummax()
    df["effective_drawdown"] = df["effective_cum_pnl"] - df["effective_running_max"]
    df["effective_rolling_20_trade_pnl"] = df[effective_pnl_col].rolling(20).sum()

    total_effective_pnl = float(df[effective_pnl_col].sum())
    total_actual_dte = float(df["actual_dte"].sum())

    if signal_col is not None and signal_col in df.columns:
        avg_signal_z = float(df[signal_col].mean())
        max_signal_z = float(df[signal_col].max())
        min_signal_z = float(df[signal_col].min())
    else:
        avg_signal_z = np.nan
        max_signal_z = np.nan
        min_signal_z = np.nan

    if "overlay_hit" in df.columns:
        overlay_hit_count = int(df["overlay_hit"].sum())
        overlay_hit_rate = float(df["overlay_hit"].mean())
    else:
        overlay_hit_count = 0
        overlay_hit_rate = np.nan

    return {
        "label": label,
        "trade_count": trade_count,
        "trade_frequency": float(trade_count / TRADE_FREQUENCY_DENOMINATOR),
        "win_rate": float(df["win"].mean()) if "win" in df.columns else np.nan,
        "avg_size_multiplier": float(df[size_col].mean()) if size_col in df.columns else 1.0,
        "notional_equiv_trade_count": float(df[size_col].sum()) if size_col in df.columns else float(trade_count),
        "avg_effective_pnl_per_day": float(df["effective_pnl_per_day"].mean()),
        "aggregate_effective_pnl_per_day": float(total_effective_pnl / total_actual_dte) if total_actual_dte else np.nan,
        "total_effective_pnl": total_effective_pnl,
        "max_drawdown": float(df["effective_drawdown"].min()),
        "worst_20_trade_sum": float(df["effective_rolling_20_trade_pnl"].min()),
        "avg_selected_tenor": float(df["selected_tenor"].mean()),
        "avg_signal_z": avg_signal_z,
        "max_signal_z": max_signal_z,
        "min_signal_z": min_signal_z,
        "overlay_hit_count": overlay_hit_count,
        "overlay_hit_rate": overlay_hit_rate,
    }


def sample_dict_for_overlay(df):
    return {
        "full_sample": df,
        "core": df[df["is_core"]],
        "secondary": df[df["is_secondary"]],
        "2026_all": df[df["is_2026"]],
        "2026_core": df[df["is_2026"] & df["is_core"]],
        "2026_Q1_core": df[df["is_2026_q1"] & df["is_core"]],
        "2026_Feb_Mar_core": df[df["is_2026_feb_mar"] & df["is_core"]],
    }


def add_summary_rows(rows, scenario_name, candidate_signal, action, threshold, df, signal_col):
    for sample_name, sample_df in sample_dict_for_overlay(df).items():
        row = summarize_effective_pnl_v2(
            sample_df,
            label=sample_name,
            signal_col=signal_col,
        )
        row.update({
            "scenario": scenario_name,
            "candidate_signal": candidate_signal,
            "signal_col": signal_col,
            "action": action,
            "threshold": threshold,
            "sample": sample_name,
        })
        rows.append(row)


summary_rows = []
trade_detail_frames = []
hit_trade_frames = []

for signal_col in SECOND_PASS_SIGNAL_COLS:
    signal_trades = trades_base_second.dropna(subset=[signal_col]).copy()
    signal_trades = signal_trades.sort_values("date").reset_index(drop=True)

    # Signal-specific baseline, so every overlay is compared to the same available universe.
    baseline = signal_trades.copy()
    baseline["candidate_signal"] = signal_col
    baseline["scenario"] = f"baseline_no_overlay__{signal_col}"
    baseline["action"] = "none"
    baseline["threshold"] = np.nan
    baseline["size_multiplier"] = 1.0
    baseline["overlay_hit"] = False
    baseline["effective_pnl"] = baseline["normalized_pnl_dollars"]

    add_summary_rows(
        rows=summary_rows,
        scenario_name=baseline["scenario"].iloc[0],
        candidate_signal=signal_col,
        action="none",
        threshold=np.nan,
        df=baseline,
        signal_col=signal_col,
    )

    trade_detail_frames.append(baseline)

    for threshold in SECOND_PASS_THRESHOLDS:
        scenario_filter = f"hard_filter__{signal_col}__z_gt_{threshold:.2f}"

        hard_filter_all = signal_trades.copy()
        hard_filter_all["candidate_signal"] = signal_col
        hard_filter_all["scenario"] = scenario_filter
        hard_filter_all["action"] = "no_trade_if_z_above_threshold"
        hard_filter_all["threshold"] = float(threshold)
        hard_filter_all["overlay_hit"] = hard_filter_all[signal_col] > threshold

        blocked = hard_filter_all[hard_filter_all["overlay_hit"]].copy()
        kept = hard_filter_all[~hard_filter_all["overlay_hit"]].copy()

        kept["size_multiplier"] = 1.0
        kept["effective_pnl"] = kept["normalized_pnl_dollars"]

        add_summary_rows(
            rows=summary_rows,
            scenario_name=scenario_filter,
            candidate_signal=signal_col,
            action="no_trade_if_z_above_threshold",
            threshold=float(threshold),
            df=kept,
            signal_col=signal_col,
        )

        trade_detail_frames.append(kept)

        if not blocked.empty:
            blocked["size_multiplier"] = 0.0
            blocked["effective_pnl"] = 0.0
            hit_trade_frames.append(blocked)

        scenario_reduce = f"reduce_50pct__{signal_col}__z_gt_{threshold:.2f}"

        reduced = signal_trades.copy()
        reduced["candidate_signal"] = signal_col
        reduced["scenario"] = scenario_reduce
        reduced["action"] = "reduce_size_if_z_above_threshold"
        reduced["threshold"] = float(threshold)
        reduced["overlay_hit"] = reduced[signal_col] > threshold
        reduced["size_multiplier"] = np.where(
            reduced["overlay_hit"],
            SECOND_PASS_REDUCED_SIZE_MULTIPLIER,
            1.0,
        )
        reduced["effective_pnl"] = reduced["normalized_pnl_dollars"] * reduced["size_multiplier"]

        add_summary_rows(
            rows=summary_rows,
            scenario_name=scenario_reduce,
            candidate_signal=signal_col,
            action="reduce_size_if_z_above_threshold",
            threshold=float(threshold),
            df=reduced,
            signal_col=signal_col,
        )

        trade_detail_frames.append(reduced)

        reduced_hits = reduced[reduced["overlay_hit"]].copy()
        if not reduced_hits.empty:
            hit_trade_frames.append(reduced_hits)


second_pass_summary = pd.DataFrame(summary_rows)
second_pass_trade_detail = pd.concat(trade_detail_frames, ignore_index=True) if trade_detail_frames else pd.DataFrame()
second_pass_hit_trades = pd.concat(hit_trade_frames, ignore_index=True) if hit_trade_frames else pd.DataFrame()

# Add deltas versus signal-specific baseline by sample.
baseline_rows = (
    second_pass_summary[second_pass_summary["action"].eq("none")]
    .set_index(["candidate_signal", "sample"])
)

for metric in [
    "trade_count",
    "trade_frequency",
    "win_rate",
    "notional_equiv_trade_count",
    "avg_effective_pnl_per_day",
    "aggregate_effective_pnl_per_day",
    "total_effective_pnl",
    "max_drawdown",
    "worst_20_trade_sum",
]:
    second_pass_summary[f"{metric}_minus_baseline"] = np.nan

for idx, row in second_pass_summary.iterrows():
    key = (row["candidate_signal"], row["sample"])
    if key not in baseline_rows.index:
        continue

    base = baseline_rows.loc[key]

    for metric in [
        "trade_count",
        "trade_frequency",
        "win_rate",
        "notional_equiv_trade_count",
        "avg_effective_pnl_per_day",
        "aggregate_effective_pnl_per_day",
        "total_effective_pnl",
        "max_drawdown",
        "worst_20_trade_sum",
    ]:
        second_pass_summary.loc[idx, f"{metric}_minus_baseline"] = row[metric] - base[metric]

second_pass_summary.to_csv(VOV_SECOND_PASS_SUMMARY_PATH, index=False)
second_pass_trade_detail.to_csv(VOV_SECOND_PASS_TRADES_PATH, index=False)
second_pass_hit_trades.to_csv(VOV_SECOND_PASS_HIT_TRADES_PATH, index=False)

print("Second-pass vol-of-vol overlay bakeoff complete")
print("  Summary path:    ", VOV_SECOND_PASS_SUMMARY_PATH)
print("  Trade detail:    ", VOV_SECOND_PASS_TRADES_PATH)
print("  Hit trades path: ", VOV_SECOND_PASS_HIT_TRADES_PATH)

# ---------------------------------------------------------------------
# Displays focused on the research question
# ---------------------------------------------------------------------

summary_display_cols = [
    "candidate_signal",
    "action",
    "threshold",
    "sample",
    "trade_count",
    "trade_frequency",
    "win_rate",
    "avg_size_multiplier",
    "notional_equiv_trade_count",
    "avg_effective_pnl_per_day",
    "aggregate_effective_pnl_per_day",
    "total_effective_pnl",
    "max_drawdown",
    "worst_20_trade_sum",
    "overlay_hit_count",
    "overlay_hit_rate",
    "avg_signal_z",
    "max_signal_z",
    "trade_count_minus_baseline",
    "win_rate_minus_baseline",
    "total_effective_pnl_minus_baseline",
    "max_drawdown_minus_baseline",
    "worst_20_trade_sum_minus_baseline",
]

print("\nFull-sample hard-filter candidates, sorted by max drawdown improvement:")
display(
    second_pass_summary[
        second_pass_summary["sample"].eq("full_sample") &
        second_pass_summary["action"].eq("no_trade_if_z_above_threshold")
    ][summary_display_cols]
    .sort_values(
        ["max_drawdown_minus_baseline", "worst_20_trade_sum_minus_baseline", "avg_effective_pnl_per_day_minus_baseline"],
        ascending=[False, False, False],
    )
    .head(30)
)

print("\n2026 Core hard-filter candidates, sorted by worst-20/max-DD improvement:")
display(
    second_pass_summary[
        second_pass_summary["sample"].eq("2026_core") &
        second_pass_summary["action"].eq("no_trade_if_z_above_threshold")
    ][summary_display_cols]
    .sort_values(
        ["worst_20_trade_sum_minus_baseline", "max_drawdown_minus_baseline", "total_effective_pnl_minus_baseline"],
        ascending=[False, False, False],
    )
    .head(30)
)

print("\n2026 Feb-Mar Core hard-filter candidates, sorted by total P&L and hit rate:")
display(
    second_pass_summary[
        second_pass_summary["sample"].eq("2026_Feb_Mar_core") &
        second_pass_summary["action"].eq("no_trade_if_z_above_threshold")
    ][summary_display_cols]
    .sort_values(
        ["total_effective_pnl_minus_baseline", "worst_20_trade_sum_minus_baseline", "overlay_hit_rate"],
        ascending=[False, False, False],
    )
    .head(30)
)

print("\n2026 Core size-reduction candidates, sorted by worst-20/max-DD improvement:")
display(
    second_pass_summary[
        second_pass_summary["sample"].eq("2026_core") &
        second_pass_summary["action"].eq("reduce_size_if_z_above_threshold")
    ][summary_display_cols]
    .sort_values(
        ["worst_20_trade_sum_minus_baseline", "max_drawdown_minus_baseline", "total_effective_pnl_minus_baseline"],
        ascending=[False, False, False],
    )
    .head(30)
)

print("\nWorst hit trades across second-pass overlays:")
if not second_pass_hit_trades.empty:
    hit_display_cols = [
        "scenario",
        "candidate_signal",
        "threshold",
        "date",
        "layer",
        "selected_tenor",
        "tenor_group",
        "win",
        "normalized_pnl_dollars",
        "pnl_per_day",
        "custom_vix30",
        "vvix_close",
    ] + SECOND_PASS_SIGNAL_COLS

    existing_hit_display_cols = [c for c in hit_display_cols if c in second_pass_hit_trades.columns]

    display(
        second_pass_hit_trades
        .sort_values("normalized_pnl_dollars", ascending=True)
        [existing_hit_display_cols]
        .head(40)
    )
else:
    print("No overlay hits found.")
